# Session 15 · Reasoning prompts
## When thinking helps, and what it costs
**A loan-eligibility checker, asked four ways — plain · step by step · step back · vote — and the price of each**

Session 14 was about *what* you say to a model. Today is about *whether to ask it to think*. You build a loan-eligibility checker for ten applicants, ask the same question four ways, and for each way read three numbers: how often is it right, what did it cost in tokens and seconds, and what would a thousand of these cost?

**How to use this notebook.** It has three parts.

| Part | What it is | Code? |
|---|---|---|
| **Part A · Read before you run** | Why this topic matters, the words you need, and how today's session is built | No code. Read it first (about 15 minutes). |
| **Part B · Hands-on** | Sections 1 to 13. Every section follows the same six steps (below). | Yes |
| **Part C · Wrap-up** | Summary, interview questions, troubleshooting | No |

Every section in Part B follows the same six steps, so you always know where you are:

1. **What and why** — what we are about to do and why it matters.
2. **The idea** — the concept, with an analogy and often a small diagram.
3. **What the next cell does** — a numbered walk-through *before* you see the code.
4. **The code** — run it.
5. **What you should see** — how to read the output, and what it means if yours is different.
6. **Check yourself** — one short question and a common mistake to avoid.

Boxes marked **▸ Read more** are optional. Click to open them when you want the deeper background; you can skip them in class.

**Before starting:** `git pull`, `uv sync` (no new packages). Please do not run the notebook the evening before class: a free key has a daily limit, and you will need it in the lab.

---
# Part A · Read before you run

## A1. Why this session matters

### The problem, in one story
A retail bank puts an AI assistant into its personal-loan app. The assistant reads an application and says **approve** or **decline** against a one-line policy: total EMIs at most 40% of income, credit score 700 or more, and age at the *end* of the loan at most 60. It is fast and cheap, because the prompt says *"reply with the answer only"*. Then three things happen in the first quarter:

1. **A quiet rule is skipped.** Mr Rao is 54 and asks for a 7-year loan. His income and score are fine, and the assistant says *approve* in one line. But 54 + 7 = 61, so the policy says *decline*. Nobody can see *why* it approved, because the prompt said "no working". The case is found months later, in an audit.
2. **The fix becomes the next problem.** A developer adds *"think step by step"* to every prompt in the bank: the loan checker, the complaint router, the SMS writer. The loan checker gets better. The monthly bill grows several times over, because every reply is now long. And the complaint router still files UPI failures in the wrong queue, because the bank's own rule ("UPI problems are `account`") was never in the prompt. More thinking about a rule you were never told gives you a more confident wrong answer.
3. **The working leaks.** To be "transparent", the team stores the model's working in a log that the customer-support team can read. One entry says: *"internal risk score 612 is below the cutoff 650, so decline, but do not tell the customer."* The SMS to the customer was clean. The log was not.

None of these is a problem with the model. **They are problems with *when* and *how* we asked it to think, and with not measuring what the thinking bought.** That is today's topic.

### What you learn today, and what each one prevents

| You learn to... | It prevents... | The everyday name |
|---|---|---|
| Ask for **step-by-step working** where several rules chain | Skipped steps on a rule that fails quietly | Show your working |
| Measure every way of asking: **right or wrong, tokens, seconds, cost per 1,000** | Paying for thinking that buys nothing | Price it before you buy it |
| Ask for the **principles first** (step back), then solve | Applying the wrong rule or formula | The pilot's checklist |
| Ask several times and take a **vote** | One unlucky random slip on a costly decision | Five underwriters, one committee |
| Turn the **reasoning-effort dial** | Paying for a deep review on an easy file | How long the underwriter spends |
| Keep the working **behind a curtain** (`<scratchpad>` and `<final>`) | Confidential working reaching a customer or a log | The back of the envelope |
| Recognise when thinking **cannot** help | A missing house rule, or arithmetic a calculator does exactly | Teach the rule; use a calculator |

### Where this is used in industry
- **Banking (BFS):** credit pre-screening and eligibility explanations, where every decision must follow written policy and be checked against an answer key before go-live. The working may be kept for internal audit, but it is not shown to the customer.
- **Insurance:** claim calculations with rules that must be applied in order (room-rent caps, deductibles, co-pay). Doing the steps in the wrong order gives a believable but wrong payout; section 9 has one.
- **Healthcare:** dose and eligibility checks. Teams use the model to explain, and a deterministic calculator or rules engine to compute. Section 9 has a teaching example with an invented formulary (not medical advice).
- **Retail and telecom:** billing and proration questions at very high volume, where the cost per 1,000 questions decides which way of asking you can afford.
- **Every LLM platform team:** model providers now sell *reasoning models* that think before they answer and bill that thinking as output tokens. Several of them offer an effort setting. Choosing the model, the effort and the prompt is now a cost decision, not only a quality decision.

### Why interviewers care
These questions come up in AI-engineer and GenAI-developer interviews:
- *"What is chain-of-thought prompting, and when does it not help?"*
- *"What are reasoning tokens, and how do they change cost and latency?"*
- *"Explain self-consistency. When would you use it, and what does it cost?"*
- *"Would you add 'think step by step' to every prompt in production?"*
- *"How do you stop a model's reasoning from leaking confidential information?"*

By the end of today you can answer each one with a definition, an example from your own lab, and one honest limit. Part C has model answers.

<details><summary><b>▸ Read more: where these techniques come from</b></summary>

All four ways of asking come from research papers that you will see cited in job descriptions and interviews. *Chain-of-thought prompting* was described by researchers at Google in 2022 (Wei and colleagues): showing worked reasoning in the prompt made large models much better at multi-step arithmetic and logic. *Self-consistency* (Wang and colleagues, 2022) added the vote: sample several chains of thought and keep the most common final answer. *Step-back prompting* (Zheng and colleagues, Google DeepMind, 2023) asks the model to state the general principle before it solves the specific case.

Since then, *reasoning models* have moved the thinking inside the model: they are trained to produce long hidden working before the visible answer. That changes the old advice. "Think step by step" was a big win on older models; on a model that already thinks, it may add cost and little else. This is exactly why today's lab **measures** instead of repeating the advice.
</details>

## A2. Where this session fits in the course

```
 S11  Multi-provider API calls ► .env, the OpenAI-compatible client, tokens and usage
  │
 S12  Local and open models ───► your local Llama (Ollama): section 8 compares it with the hosted model
  │
 S14  Prompt anatomy ──────────► the five parts of a prompt, Tasks 1-3, the complaint classifier,
  │                              and prompts/library.md with your version 1 entries
 S15  Reasoning prompts ───────► TODAY: plain vs step by step vs step back vs vote; reasoning tokens;
  │                              the effort dial; the scratchpad curtain; Lab, DiskCache and Meter
 S16  Structured prompting ────► tags, templates, a system prompt in a file; reuses today's Lab,
  │                              DiskCache (free re-runs) and Meter (token counts)
 S17  Prompt evaluation ───────► a judge scores your v1 against your v2: keep both!
  │
 S18  Defensive prompting ─────► attacks on prompts; today's scratchpad leak is a first taste
  │
 S19  Chat architecture ───────► a token budget per turn, using today's habit of metering every call
  │
 S20  Function calling ────────► give the model a calculator for the EMI instead of paying for votes
  │
 S34  What is an agent ────────► the ReAct loop: reasoning written out, then an action, then repeat
 S53–54  Observability, LLMOps ► cost per run, routing between models, caching, prompt versions at scale
```

**What you need from earlier sessions:**
- A working `.env` in the course folder with `BASE_URL`, `API_KEY` and `MODEL` (Session 11). Without it the model sections say "skipped"; everything else still runs.
- Your **work repository** from Session 14 at `Documents\genai-course-work`, with `prompts/library.md` holding your version 1 entries for Tasks 1 and 2. Today you add version 2 *beside* them.
- Optional: Ollama with a small chat model from Session 12, for section 8.

## A3. The words you need today

Read these once now. Each term appears in bold the first time it is used in Part B.

---
**1. Reasoning model**
- **Definition:** a model that writes hidden working before its visible answer. You pay for that working as output tokens, whether you read it or not.
- **Analogy:** a loan officer who thinks through the file in silence before saying "approved". The thinking took time, and the bank paid for that time.
- **Example:** the hosted model you have used since Session 2 is a reasoning model. Section 2 shows a one-line answer that cost far more than one line.
- **Often confused with:** a "smarter" model. It is a model that spends more tokens before answering. Whether that helps *your* question is something you measure.
- **Comes back in:** S17, S54.

---
**2. Reasoning tokens (thinking tokens)**
- **Definition:** the output tokens spent on the hidden working. They are counted inside the output (completion) total.
- **Analogy:** the tokens you pay for and never read.
- **Example:** a reply of one visible line can report a few hundred completion tokens, most of them reasoning tokens.
- **Often confused with:** *prompt* tokens, which are what you send. Reasoning tokens are what the model writes. When a provider does not report them, the notebook either estimates them from the thinking text (about 4 characters per token) or shows a dash.
- **Comes back in:** S19, S53.

---
**3. Chain of thought (CoT, "step by step")**
- **Definition:** asking the model to write its steps before the answer.
- **Analogy:** a loan officer who writes the calculation on the file instead of only "looks fine". When the file is audited, the working is what you can check.
- **Example:** *"Think step by step and show each calculation. Then end with one last line in the form FINAL: &lt;answer&gt;."*
- **Often confused with:** a guarantee of correctness. The steps can be wrong too, and they cost tokens on every call.
- **Comes back in:** S16, S34.

---
**4. The FINAL line**
- **Definition:** the agreed last line of every reply, `FINAL: <answer>`, that code reads to get the answer.
- **Analogy:** the box at the bottom of a form marked "decision". The rest of the form is working.
- **Example:** `FINAL: decline`. If the model writes `FINAL:` twice (it changed its mind), the **last** one counts.
- **Often confused with:** reading the answer from the prose. Working is full of numbers and words like "approve if..."; never guess from it.
- **Comes back in:** S17, S20.

---
**5. Ground truth (the answer key)**
- **Definition:** the one correct answer for each case, worked out independently of the model, that every reply is checked against.
- **Analogy:** the answer key a teacher marks with.
- **Example:** for case L05 the key says `decline`, because 54 + 7 = 61. Python computed it from the policy, and the tests check it again by hand.
- **Often confused with:** what the model said most often. A majority can be wrong; the key cannot argue.
- **Comes back in:** S17, S52.

---
**6. Step-back prompting**
- **Definition:** two calls. First ask for the general rules, formulas or traps for *this kind* of problem, in a call that is **not allowed to solve**. Then solve, with those principles in front of the model.
- **Analogy:** the pilot's pre-flight checklist: read the checklist for this type of flight, then fly this one.
- **Example:** call 1: *"Do NOT solve this yet. Step back: list ... the general rules, formulas or traps."* Call 2: the case, the rules, step by step, FINAL.
- **Often confused with:** chain of thought. CoT works the case; step back first writes the checklist, then works the case.
- **Comes back in:** S16.

---
**7. Self-consistency (voting)**
- **Definition:** ask the same step-by-step question several times, at a temperature above zero, and take the most common final answer.
- **Analogy:** five underwriters look at the same file independently, and the committee goes with the majority.
- **Example:** five samples say decline, approve, decline, decline, approve: the vote is *decline*, 3 of 5.
- **Often confused with:** asking five times at temperature 0. Then the five answers are (nearly) the same one, and you paid five times for one opinion.
- **Comes back in:** S17, S52.

---
**8. Temperature above zero, and samples**
- **Definition:** temperature adds randomness to how the model picks each word. At 0 it picks (almost) the same reply every time; above 0, replies can differ. Each separate draw is a *sample*.
- **Analogy:** a little noise on purpose, so the five underwriters do not copy each other.
- **Example:** the vote uses temperature 0.8 and samples numbered 0 to 4, so the memory (DiskCache) keeps five different answers.
- **Comes back in:** S11 (where you swept it), S17.

---
**9. Majority vote and agreement**
- **Definition:** the winning answer, and the share of valid votes it got. A tie goes to the answer seen first, so the result is the same every run.
- **Analogy:** "how many of the five underwriters agree?"
- **Example:** 3 of 5 for decline is 60% agreement. 5 of 5 is 100%, which means the model was confident, **not** that it was right.
- **Often confused with:** accuracy. Five votes that apply the same wrong rule agree perfectly on the wrong answer.
- **Comes back in:** S52.

---
**10. Reasoning effort (the dial)**
- **Definition:** a request setting on some reasoning models, `reasoning_effort`, with values `"low"`, `"medium"` or `"high"`. It trades tokens and seconds against how carefully the model works.
- **Analogy:** how long an underwriter spends on a file: skim, normal review, deep review.
- **Example:** in section 6 the same plain question is asked at low and at high; only the dial changes.
- **Often confused with:** temperature. Temperature changes randomness; effort changes how much thinking is done. Not every model or provider has the dial; the notebook says so instead of failing.
- **Comes back in:** S54.

---
**11. DiskCache (the answer memory)**
- **Definition:** a file in your home folder (`.genai_course\s15_cache.json`) that remembers every answer. The same question with the same settings is answered from disk: free, instant, identical.
- **Analogy:** a photocopy of every answer you have already paid for.
- **Example:** re-running section 3 costs zero tokens the second time, and the table shows the same numbers.
- **Often confused with:** the model "remembering". The model remembers nothing; your code looks up the old reply before calling.
- **Comes back in:** S16, S22, S54.

---
**12. Meter and token budget**
- **Definition:** the *Meter* adds up the calls and tokens this notebook has spent. The *token budget* is what you may spend: per minute, per day, per month.
- **Analogy:** the taxi meter, next to the cash in your wallet.
- **Example:** `spent so far: 21 calls (0 answered from memory), 15,320 tokens = about 8% of a 200,000-token free daily limit` (an illustration, not a real run).
- **Comes back in:** S19, S53, S54.

---
**13. Rate limit (429) and daily limit**
- **Definition:** a provider's "slow down" reply, HTTP status 429. A per-minute limit is waited out; a daily limit means stop for today.
- **Analogy:** a queue at the bank counter (wait a minute) versus the branch being closed (come back tomorrow).
- **Example:** `(rate limit: waiting 21s, then trying again)` is normal. A message that says "That is a daily limit" means stop; answers already received are saved.
- **Comes back in:** S22, S54.

---
**14. Cost per 1,000 questions**
- **Definition:** tokens times price, scaled to 1,000 questions, so that two ways of asking can be compared in money.
- **Analogy:** the number finance asks for.
- **Example:** the price table in `reasoning_utils.py` (Groq, `openai/gpt-oss-20b`, checked October 2026, verify before quoting) is $0.075 per million input tokens and $0.30 per million output tokens.
- **Often confused with:** cost per call. A fraction of a cent per call looks like nothing until you multiply by 200,000 decisions a month.
- **Comes back in:** S53, S54.

---
**15. Scratchpad and `<final>`**
- **Definition:** the model writes its working inside `<scratchpad>...</scratchpad>` and the real reply inside `<final>...</final>`. Your code keeps only the `<final>` part to check, show and store.
- **Analogy:** the back of the envelope, and the clean letter you post.
- **Example:** the scratchpad says "score 612 is below the cutoff"; the final SMS says only "not approved, apply again after 90 days".
- **Often confused with:** a security feature. The scratchpad is *more* dangerous, not less: it is exactly where confidential facts appear. It must never be shown or stored where a customer can read it.
- **Comes back in:** S18, S34.

---
**16. Prompt version 2**
- **Definition:** a new entry in `prompts/library.md`, added beside version 1, with a one-line reason **in numbers**. Version 1 is never edited.
- **Analogy:** a new edition of a manual; the old edition stays on the shelf.
- **Example:** `"4.0 vs 3.0 of 4 checks, 2.1x tokens, worth it for a credit decision"`.
- **Comes back in:** S17 (a judge scores v1 against v2), S53.

## A4. How this session is built

### The big picture

```
┌── COURSE REPOSITORY (you pull it; you do not edit it) ─────────────────────────────────────────────────────┐
│                                                                                                            │
│  module02/reasoning_tasks.py   THE PROBLEMS (synthetic people, fake numbers)                               │
│     DEMO       the jacket warm-up (section 2)      LOAN_POLICY  the one-line policy                        │
│     LOAN_CASES ten applicants, each with a trap; 5 approve, 5 decline; truth worked out by Python          │
│     OTHER      retail, insurance, factory puzzles   STRETCH  dosage, proration, EMI (section 9)            │
│     classify_cot_spec · score_classifier_lab · problem_card                                                │
│                                                                                                            │
│  module02/reasoning_utils.py   THE TOOLS                                                                   │
│     Lab (talks to the model) · DiskCache (free re-runs) · Meter (tokens) · run_strategy (4 ways)           │
│     case_table · strategy_summary · comparison_table · pick_trouble · pick_hardest                         │
│     with_scratchpad · pass_rate · verdict · accuracy_markdown · save_accuracy_table · check_s15_work       │
│                                                                                                            │
│  module02/prompt_tasks.py + prompt_utils.py (S14)  TASK1, TASK2, classify_spec, save_to_library            │
│  module01/providers_utils.py (S11)                reads your .env; groq_client, ollama_client              │
│                                                                                                            │
│  module02/s15_reasoning.ipynb  ◄── YOU ARE HERE                                                            │
│  tests/test_s15.py             the checkpoint (pytest)    reading_reasoning.md · terms_s15.md              │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
            │ the notebook calls the model through Lab, and reads and writes ▼
┌── YOUR MACHINE (yours; never committed) ───────────────────────────────────────────────────────────────────┐
│  .env                              BASE_URL, API_KEY, MODEL (Session 11)                                   │
│  ~\.genai_course\s15_cache.json    DiskCache: every answer you have paid for, so re-runs are free          │
│  Ollama (optional)                 your local Llama from Session 12, for section 8                         │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
┌── YOUR WORK REPOSITORY  Documents\genai-course-work (yours; you commit and push it) ───────────────────────┐
│  prompts/library.md                v1 entries from Session 14 stay untouched; today adds v2 for T1, T2     │
│  experiments/s15_accuracy_table.md the ten cases x the ways you asked: THE CHECKPOINT                      │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

### What happens to one loan case

```
 one loan case, e.g. L05: "Suresh Rao, age 54, income Rs 90,000, EMIs 14,000 + 21,500, 7 years, score 731"
        │
        ▼  run(case, strategy)  ──►  run_strategy(lab, case, strategy)
        │
        ├── plain ─────► 1 call : the case + "Reply with only ... FINAL: <answer> ... Do not show any working."
        ├── cot ───────► 1 call : the case + "Think step by step and show each calculation. Then ... FINAL:"
        ├── stepback ──► 2 calls: (1) "Do NOT solve this yet. Step back: list ... rules ... traps"
        │                         (2) the case + those rules + "Think step by step ... FINAL:"
        └── vote x5 ───► 5 calls: the cot question at temperature 0.8, samples 0..4; the majority wins
                              │
                              ▼  every call goes through Lab.chat
              asked before with the same settings? ── yes ──► answer from DiskCache, 0 tokens
                              │ no
                              ▼
              the model (Groq) ──► text + prompt / completion / reasoning tokens + seconds
                              │       (429: wait and retry · daily limit: stop, nothing is lost)
                              ▼  Meter adds the tokens
              extract_final: the LAST "FINAL: ..." line ──► parse_answer ──► "decline"
                              │
                              ▼  is_correct: compare with the answer key (L05: decline)
              Attempt: answer, right/wrong, calls, tokens, thinking tokens, seconds
                              │
                              ▼  results[(case id, strategy)] = attempt
              case_table · strategy_summary ──► accuracy_markdown ──► your work repository
```

### The helpers you will call, and what each does

| Helper | Lives in | You give it | It gives back | Used in section |
|---|---|---|---|---|
| `groq_client()`, `ollama_client()` | providers_utils (S11) | nothing (reads `.env`) | a client for Groq, or for local Ollama | 1, 8 |
| `DiskCache()`, `Meter()` | reasoning_utils | nothing | the answer memory, and a token counter; `meter.report()` prints what you spent | 1, every model section |
| `Lab(client, model, cache=, meter=)` | reasoning_utils | a client and model name | an object whose `chat()` makes one measured, cached call | 1, 8 |
| `lab.variant(effort=...)` | reasoning_utils | one setting to change | the same lab with that setting; cache and meter shared | 6 |
| `run(problem, strategy)` | **this notebook** (section 1) | a problem and `"plain"`, `"cot"`, `"stepback"` or `"vote"` | an `Attempt`, also stored in `results` | 2–5, 9 |
| `rows(problems, strategies)` | **this notebook** (section 1) | which cases, which strategies | the stored attempts, in order | 3–5, 9 |
| `run_strategy(lab, problem, strategy)` | reasoning_utils | a lab, a problem, a strategy name | an `Attempt` (answer, correct, calls, tokens, thinking tokens, seconds, notes) | 1, 6, 8 |
| `problem_card(p)` | reasoning_tasks | a problem | its id, domain, title and text, ready to print | 2 |
| `case_table(results, problems, strategies)` | reasoning_utils | the results and which columns | one row per case: the right answer, what each strategy said, ok or WRONG | 3, 4 |
| `strategy_summary(attempts, model)` | reasoning_utils | attempts and the model name | one row per strategy: right, average tokens, seconds, $ per 1,000 | 3, 4, 8 |
| `comparison_table(attempts, model)` | reasoning_utils | attempts | one row per attempt, with calls, tokens, thinking, seconds, $ per 1,000 | 6, 8, 9 |
| `pick_trouble(attempts, problems, 3)` | reasoning_utils | results so far, the cases | the 3 cases answered wrongly most often (ties: most tokens) | 4 |
| `pick_hardest(attempts, problems)` | reasoning_utils | results so far, the cases | the single most troublesome case | 5 |
| `classify_spec(n)` | prompt_tasks (S14) | 0 or 4 examples | the Session 14 classifier prompt | 7 |
| `classify_cot_spec()` | reasoning_tasks | nothing | the same classifier, asked to think first and end with `FINAL:` | 7 |
| `score_classifier_lab(lab, spec, mode=)` | reasoning_tasks | a lab, a prompt, `"plain"` or `"cot"` | right out of 8, tokens, seconds, the wrong ones | 7 |
| `pick_local_chat_model(names)` | reasoning_utils | the local model names | the first that is not an embedding model, or `None` | 8 |
| `with_scratchpad(spec)` | reasoning_utils | a Session 14 prompt | a copy that asks for `<scratchpad>` then `<final>` | 10, 11 |
| `pass_rate(lab, task, spec, hidden_reasoning=, runs=)` | reasoning_utils | a lab, a task, a prompt | average checks passed, tokens, and whether the scratchpad leaked | 10 |
| `verdict(v1, v2)` | reasoning_utils | two `pass_rate` results | one sentence: better, worse or the same, and at what token ratio | 10 |
| `save_to_library(...)` | prompt_utils (S14) | your repo, task, version, prompt, output, checks | appends one entry to `prompts/library.md` | 11 |
| `accuracy_markdown(...)`, `save_accuracy_table(repo, text)` | reasoning_utils | the results | the ten-case table as Markdown; writes it to `experiments/` | 11 |
| `check_s15_work(repo)` | reasoning_utils | your work repo path | five PASS/FAIL checks, each with a fix | 11 |

You do not need to read `reasoning_utils.py`. Where a helper matters most, the section has an **under the hood** cell that rebuilds a short version of it in front of you, and checks that it gives the same answer as the real one.

### The order of the sections

| # | Section | Calls the model? | Minutes |
|---|---|---|---|
| 1 | Setup | no | 3 |
| 2 | The surprise: a one-line answer that costs hundreds of tokens | yes, 1 call | 5 |
| 3 | The loan-eligibility checker: ten cases, two ways of asking | yes, 20 calls | 10 |
| 4 | Step back | yes, 6 calls (3 cases × 2) | 5 |
| 5 | Take a vote (self-consistency) | yes, 5 calls | 4 |
| 6 | The dial: reasoning effort | yes, 2 calls | 3 |
| 7 | When thinking does not help | yes, 24 calls (8 messages × 3) | 4 |
| 8 | A different kind of model: your local Llama | local only, 10 calls, no tokens billed | runs in the background |
| 9 | Other domains (optional) | off by default; 12 calls if you switch it on | optional |
| 10 | Reasoning behind a curtain | yes, 8 calls (2 tasks × 2 versions × 2 runs) | 6 |
| 11 | Your work repository: the accuracy table and prompt library v2 | no | 6 |
| 12 | Your decision table | no | 5 |
| 13 | Reflection | no | 5 |

About 66 hosted calls in all, plus 12 if you switch on section 9.

## A5. What you will be able to do, and what you will submit

**By the end of this session you will be able to:**
1. Ask the same question four ways (plain, step by step, step back, vote) and **measure** each: right or wrong, tokens, seconds, cost per 1,000.
2. Explain what reasoning tokens are, who pays for them, and why "think step by step" may add little on a model that already thinks.
3. Decide, from your own numbers, which kinds of question deserve thinking, and which do not (a missing rule, an exact calculation).
4. Use the `reasoning_effort` dial, and handle a model or provider that does not have one.
5. Hide the model's working behind a curtain, check only the `<final>` part, and explain why the scratchpad must never be shown or openly stored.
6. Add a version 2 prompt to your library beside version 1, with a reason in numbers.

**What you submit (in your work repository):**

| File | What it must contain |
|---|---|
| `experiments/s15_accuracy_table.md` | All ten loan cases, each strategy you ran marked `ok` or `WRONG` against the answer key, a totals row and average tokens. **This is the session checkpoint.** |
| `prompts/library.md` | Your Session 14 version 1 entries, untouched, **plus** version 2 entries for Task 1 and Task 2 (the scratchpad version), each with a one-line reason in numbers |

You also fill in the **decision table** in section 12 and answer the reflection in section 13.

**The checkpoint:** `uv run pytest tests/test_s15.py` is green, and the self-check in section 11 says `PASS` on all five lines. The two live tests at the bottom of the test file skip when there is no key (or no Ollama). *A skipped test is not a pass:* run the tests with your key in `.env`, so the Groq live test really runs. The Ollama test is optional, like section 8.

**Budget:** somewhere between 40,000 and 80,000 of the 200,000 tokens a free Groq key gets per day (an estimate; the meter under each section shows your real number). The free plan allows about 8,000 tokens a minute, so some cells will pause and say they are waiting for the rate limit. That is normal. Every answer is remembered on disk, so **re-running a cell is free and gives the same numbers**; delete the memory only when you want a fresh draw. Numbers vary by model and by day: read your own tables.

---
# Part B · Hands-on

## 1. Setup

### What and why
Before anything else, the notebook must find the course's helper files, read your `.env`, open the answer memory, and connect to the model. If any of this fails, it should fail *now*, with a clear message, rather than in the middle of the lab. This cell also defines two small helpers, `run` and `rows`, that every later section uses to keep the results in one place.

### The idea
Think of opening a bank branch in the morning: unlock the door (find the files), switch on the system (read your key), open the filing cabinet (the DiskCache with yesterday's answers), and reset the meter on the till (the Meter). If the system is down, the branch still opens: you can do the paperwork, meaning the sections that do not call the model, while you wait.

Every result of the day goes into **one dictionary**, `results`. Its keys are pairs, `(case id, strategy)`, and its values are `Attempt` objects:

```
 results = {
   ("L05-age-at-end", "plain"):   Attempt(answer="approve", correct=False, tokens=..., seconds=...),
   ("L05-age-at-end", "cot"):     Attempt(answer="decline", correct=True,  tokens=..., seconds=...),
   ("L05-age-at-end", "vote x5"): Attempt(...),
   ...
 }
```
Because the key is the pair, running a cell twice **replaces** an entry instead of adding a duplicate. The tables and the accuracy file are all built from this one dictionary.

### What the next cell does
1. **`sys.path[:0] = [...]`** puts the course folders at the front of Python's search list, so `import reasoning_utils` finds the file in `module02/` whether you opened the notebook from the repo root or from inside `module02/`.
2. **`try: ... from ... import ...`** loads the helpers: providers from Session 11, prompts and tasks from Session 14, and today's two files. Each import line has a comment saying which session it comes from.
3. **`except ImportError as e:`** catches the most common problem, a missing file, and replaces Python's message with one that tells you what to do (`git pull`, then restart the kernel). `raise ... from e` keeps the original error attached underneath, for debugging.
4. **Importing `providers_utils` reads your `.env`** (it calls `load_dotenv(override=True)`), so `os.getenv("MODEL")` now sees your settings.
5. **`cache, meter = DiskCache(), Meter()`** opens the answer memory at `~\.genai_course\s15_cache.json` and starts a fresh token counter. Two names are assigned in one line (tuple unpacking).
6. **`client = groq_client()`** runs only if all three settings (`BASE_URL`, `API_KEY`, `MODEL`) exist. Otherwise `client` stays `None`.
7. **`lab = Lab(...) if client else None`** is Python's one-line *conditional expression*: "make a lab if we have a client, otherwise `None`". **`ready`** is then `True` or `False`, and every model section checks it.
8. **`results = {}`** is the dictionary described above. **`hardest = LOAN_CASES[4]`** is a default (case L05, the fifth case, because lists count from 0); section 5 replaces it with the case that really gave the most trouble.
9. **`def run(problem, strategy, **kw):`** asks the model with one strategy and stores the result under `(problem.id, a.strategy)`. The **`**kw`** collects any extra named arguments (for example `n=5` for the vote) and passes them on unchanged to `run_strategy`.
10. **`def rows(...)`** returns the stored attempts for some cases and strategies. Its body is a *list comprehension with two `for` parts and an `if`*: for each problem, for each strategy, keep the result **if** that pair was actually run. Read it left to right like a sentence. The default values (all ten cases, all four strategies) are used when you call `rows()` with nothing.

<details><summary><b>▸ Read more: why not just stop if there is no key?</b></summary>

Because a lot of today does not need the model: the policy, the answer key, the under-the-hood cells, the scratchpad prompt, the work-repository steps and the decision table. A student whose key is missing, or who has used up the day's tokens, should still learn those. So model sections print "skipped" instead of crashing. Note that **skipped is never the same as passed**: your accuracy table needs real results, and the live test reports a skip clearly.
</details>

In [1]:
import sys; sys.path[:0] = [".", "module01", "module02", "../module01", "../module02"]
import os
from pathlib import Path
try:
    from providers_utils import groq_client, ollama_client                                   # Session 11
    from prompt_utils import PromptSpec, init_work_repo, save_to_library                      # Session 14
    from prompt_tasks import TASK1, TASK2, TASK3, classify_spec                               # Session 14
    from reasoning_tasks import (DEMO, LOAN_CASES, LOAN_POLICY, OTHER, STRETCH, classify_cot_spec, problem_card,
                                 score_classifier_lab)
    from reasoning_utils import (DiskCache, Lab, Meter, accuracy_markdown, case_table, check_s15_work, comparison_table,
                                 pass_rate, pick_hardest, pick_local_chat_model, pick_trouble, run_strategy,
                                 save_accuracy_table, strategy_summary, verdict, with_scratchpad)
except ImportError as e:
    raise ImportError(f"{e}. This notebook uses files from Sessions 11, 14 and 15: run 'git pull' in the course folder, "
                      "then restart the kernel.") from e

MODEL = os.getenv("MODEL")
cache, meter = DiskCache(), Meter()
client = None
if os.getenv("BASE_URL") and os.getenv("API_KEY") and MODEL:
    client = groq_client()
lab = Lab(client, MODEL, cache=cache, meter=meter) if client else None
ready = lab is not None
print("model:", MODEL, "| ready:", ready)
print("answers are remembered in:", cache.path)
if not ready:
    print("No key in .env: the sections that call the model will say 'skipped'. Everything else still runs.")

results = {}                       # (problem id, strategy) -> Attempt, so re-running a cell replaces rather than duplicates
hardest = LOAN_CASES[4]            # replaced in section 5 by the case that gave the most trouble

def run(problem, strategy, **kw):
    a = run_strategy(lab, problem, strategy, **kw)
    results[(problem.id, a.strategy)] = a
    return a

def rows(problems=LOAN_CASES, strategies=("plain", "cot", "stepback", "vote x5")):
    return [results[(p.id, s)] for p in problems for s in strategies if (p.id, s) in results]

model: openai/gpt-oss-20b | ready: True
answers are remembered in: C:\Users\mynam\.genai_course\s15_cache.json


### What you should see
Two or three lines, something like this (an illustration, not a real run):
```
model: openai/gpt-oss-20b | ready: True
answers are remembered in: C:\Users\you\.genai_course\s15_cache.json
```
- **`ready: True`** means the model sections will run. **`ready: False`** plus the "No key in .env" line means they will be skipped.
- The cache path shows *your* user folder. It is outside every Git repository on purpose: answers are not code.
- Nothing is printed by the two `def` lines; they only define `run` and `rows` for later.

| If you see... | It means... | Do this |
|---|---|---|
| `ImportError: ... run 'git pull'` | a course file is missing or old | `git pull`, `uv sync`, then **Kernel → Restart** |
| `ready: False` but you have a key | `.env` not found, or a name is misspelled | `.env` must sit in the course folder with `BASE_URL`, `API_KEY`, `MODEL`; restart the kernel |
| `model: None` | `MODEL` is missing from `.env` | add it (Session 11), restart the kernel |

### Check yourself
*Why does `results` use the pair `(case id, strategy)` as its key, instead of appending every attempt to a list? What would go wrong in the accuracy table if you ran section 3 twice with a list?*

> **Common mistake:** editing `.env` and re-running only this cell. Python has already read the old values. Restart the kernel after changing `.env`.

## 2. The surprise: a one-line answer that costs hundreds of tokens

### What and why
A warm-up from a shop: *a jacket costs Rs 2,000, take 20% off, then 10% off the reduced price, then a Rs 100 coupon.* We ask for **only the final answer, no working**. The visible reply will be one short line. The question is: how many tokens did that one line cost? Your guess, and the real number, are the first lesson of the day: with a reasoning model, the part you read and the part you pay for are not the same size.

### The idea: the visible answer is the tip of the iceberg
The arithmetic first, so you can check the model: 2,000 × 0.80 = 1,600; then 1,600 × 0.90 = 1,440; then 1,440 − 100 = **1,340**. (The trap: 20% then 10% is *not* 30% off, because the second discount applies to the reduced price.)

```
 what you send                     what the model produces                     what you read
 ─────────────────                 ─────────────────────────────────           ─────────────
 system: careful analyst           ┊ hidden thinking: "20% off 2000 is ┊
 user:   the jacket question  ──►  ┊ 1600, then 10% of 1600 is 160 ... ┊  ──►    FINAL: 1340
         + "Reply with only        ┊ so 1440, minus 100 ..."           ┊
         FINAL: <answer>.          ╰── billed as output tokens, often ─╯
         Do not show any               not shown to you
         working."
   = prompt tokens                  = completion tokens (reasoning tokens + the visible line)
```
**Analogy:** a loan officer says "approved" in one word, but spent twenty minutes on the file first. The bank pays for the twenty minutes, not for the word.

The exact prompt the helper sends for "plain" is: the problem text, then *"Reply with only the final answer, on one line in the form FINAL: &lt;answer&gt;, where &lt;answer&gt; is the amount in rupees, as a number. Do not show any working."* The system message is always *"You are a careful analyst. Apply the rules in the question exactly as written."*

**Predict first.** Before you run the cell, change `my_guess` to the number of *output tokens* you expect this one-line reply to cost.

### What the next cell does
1. **`print(problem_card(DEMO))`** prints the warm-up problem: its id, domain, title and text.
2. **`my_guess = 20`** is your prediction. Change it *before* running.
3. If `ready`, **`a = run(DEMO, "plain")`** makes one call and stores the `Attempt` in `results`.
4. Prints the parsed answer (`a.raw`), the right answer and whether they match. **`{DEMO.truth:g}`** is an *f-string* with a format code: `:g` prints `1340.0` as `1340`.
5. Prints the completion (output) tokens next to your guess. **`{a.completion_tokens:,}`** adds thousands separators (`1,234`).
6. If the model reported thinking tokens (`a.reasoning_tokens is not None`), splits the output into *thinking you never saw* and *the visible reply*. Otherwise it says the model reported none.
7. **`meter.report()`** prints the running total for the notebook so far.

In [2]:
print(problem_card(DEMO))
my_guess = 20          # YOUR TURN: how many OUTPUT tokens will a one-line answer cost? Change this, then run.
if ready:
    a = run(DEMO, "plain")
    print(f"answer: {a.raw}   | right answer: {DEMO.truth:g}   | correct: {a.correct}")
    print(f"output tokens: {a.completion_tokens:,}   (you guessed {my_guess})")
    if a.reasoning_tokens is not None:
        print(f"   thinking you never saw: {a.reasoning_tokens:,}      the visible reply: {a.completion_tokens - a.reasoning_tokens:,}")
    else:
        print("   (this model did not report any thinking tokens)")
    print(meter.report())
else:
    print("skipped (no key in .env)")

=== D0-jacket  (Retail): Two discounts and a coupon
A jacket is priced at Rs 2,000. The shop takes 20% off, then takes a further 10% off the REDUCED price, and then subtracts a Rs 100 coupon. What does the customer pay, in rupees?

answer: 1340   | right answer: 1340   | correct: True
output tokens: 116   (you guessed 20)
   thinking you never saw: 102      the visible reply: 14
spent so far: 0 calls (1 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


### What you should see
The jacket problem, then lines shaped like this (an illustration, not a real run; your numbers will differ):
```
answer: 1340   | right answer: 1340   | correct: True
output tokens: 214   (you guessed 20)
   thinking you never saw: 190      the visible reply: 24
spent so far: 1 call (0 answered from memory), 380 tokens = about 0% of a 200,000-token free daily limit
```
**How to read it:** compare the *output tokens* line with your guess first. Then look at the split: if most output tokens were thinking, you paid for working you never saw. Run the cell a second time: the meter now says `1 answered from memory`, and the numbers do not change, because the answer came from the DiskCache.

| If you see... | It means... | Do this |
|---|---|---|
| `(this model did not report any thinking tokens)` | the provider sent no reasoning count and no thinking text | not a bug; compare the output tokens with the length of the visible line instead |
| `answer: None` | the reply had no readable `FINAL:` line (or was cut off) | look at `a.text`; an unreadable answer is counted as wrong, never guessed |
| `(rate limit: waiting 21s, then trying again)` | the free plan's per-minute limit | wait; do not click Stop |
| `skipped (no key in .env)` | `ready` is False | see section 1 |

### Under the hood: read the FINAL line yourself
Every answer today is read by one small helper, `extract_final`. It looks for the **last** line that starts with `FINAL:` (allowing Markdown decoration such as `**` or `>`), ignores anything inside `<think>` tags, and returns the text after the colon. Then `parse_answer` turns that text into a label or a number. If there is no `FINAL:` line, the answer is `None`, and `None` is always marked wrong. That rule is what makes the scoring honest: the code never guesses from the prose.

**What the next cell does:**
1. Imports the real `extract_final` and `parse_answer`, to compare against.
2. Defines `my_extract_final(text)` in about ten lines: remove `<think>...</think>` with a regular expression (`re.sub`), then walk through the lines with a `for` loop, strip decoration from the left, and keep the text after `FINAL` and a colon. The *last* one found wins (`found[-1]`).
3. Runs both versions on eight sample replies, including tricky ones: a model that changes its mind, bold text, a `<think>` block, the word "Finally", and a reply with no FINAL line at all.
4. Prints `same` or `DIFFERENT` for each, then what `parse_answer` makes of the real result.

No model call; it runs without a key.

In [3]:
import re
from reasoning_utils import extract_final, parse_answer          # the real helpers, to compare against

def my_extract_final(text):
    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL | re.IGNORECASE)   # hidden thinking is not the answer
    found = []
    for line in text.splitlines():
        line = line.strip().lstrip(">*_`#- ")                    # Markdown decoration before FINAL
        rest = line[5:].lstrip(" \t")
        if line[:5].upper() == "FINAL" and rest.startswith(":"):
            value = rest[1:].lstrip(" \t").rstrip(" \t*_`")      # the text after the colon, without trailing decoration
            if value:
                found.append(value.strip())
    return found[-1] if found else None                          # the LAST FINAL line wins

samples = ["FINAL: approve",
           "Ratio 38%, score 705, age at end 53.\nFINAL: approve",
           "FINAL: approve\nWait: 54 + 7 = 61, which is over 60.\nFINAL: decline",
           "**FINAL: Declined**",
           "<think>FINAL: approve</think>\nFINAL: decline",
           "> final : Rs 1,340",
           "Finally: approve",
           "I would approve this loan."]
for s in samples:
    mine, real = my_extract_final(s), extract_final(s)
    print(f"{'same' if mine == real else 'DIFFERENT':<10}{s.splitlines()[-1][:34]!r:<38}-> {real!r:<12}"
          f" label: {parse_answer(real, 'label', ('approve', 'decline'))}")
print("\nall the same:", all(my_extract_final(s) == extract_final(s) for s in samples))

same      'FINAL: approve'                      -> 'approve'    label: approve
same      'FINAL: approve'                      -> 'approve'    label: approve
same      'FINAL: decline'                      -> 'decline'    label: decline
same      '**FINAL: Declined**'                 -> 'Declined'   label: decline
same      'FINAL: decline'                      -> 'decline'    label: decline
same      '> final : Rs 1,340'                  -> 'Rs 1,340'   label: None
same      'Finally: approve'                    -> None         label: None
same      'I would approve this loan.'          -> None         label: None

all the same: True


**What you should see:** eight rows that all start with `same`, then `all the same: True`. Read the right-hand columns: the model that changed its mind is judged on its **last** word (`decline`); `Declined` is read as the label `decline`; `> final : Rs 1,340` gives the text `Rs 1,340` (and no label, because it is a number); "Finally" and the plain sentence give `None`, so they would be marked wrong.

**Python notes:** `{s!r}` inside an f-string prints the value with quotes, the way you would type it (`repr`). `{x:<10}` pads to 10 characters, left-aligned; that is how every table in this notebook keeps its columns straight. `all(... for s in samples)` is `True` only if every item is true.

**The take-away:** the format instruction (`FINAL: <answer>`) and the reader are a pair. A clear, checkable answer line is what lets you score a thousand replies automatically. You will use the same idea for the golden set in Session 17.

### Check yourself
*Question 1 — Where did the output tokens go? Who paid for them, and could you see them? What does that mean for the phrase 'think step by step' on a model that already thinks?*

> **Common mistake:** judging the cost of a reply by its length on screen. On a reasoning model the bill is the **completion tokens**, and most of them can be hidden thinking.

## 3. The loan-eligibility checker: ten cases, two ways of asking

### What and why
This is the heart of the session. The policy is the same for every case. Ten applicants, each hiding a different trap: a boundary, a unit, a rule that fails quietly. The answer is always `approve` or `decline`, and Python worked out the right one independently, so every answer is marked right or wrong.

- **Plain**: "reply with only `FINAL: <answer>`, no working".
- **Step by step** (**chain of thought**), your *explainer*: "think step by step, show each calculation, then end with `FINAL: <answer>`".

Same cases, same model, same system message. **Only the instruction changes.** That makes it a fair test: if the results differ, only the instruction can explain it.

### The idea: one policy, ten traps
The policy: approve only if **all three** hold. (a) total monthly EMIs, existing plus the new one, are at most 40% of monthly income; (b) the credit score is 700 or more; (c) the age at the **end** of the loan term is at most 60. Otherwise decline.

| Case | The trap | Key |
|---|---|---|
| L01 clean | none: a control case | approve |
| L02 ratio narrow | (12,000 + 21,000) / 80,000 = 41.25%: just over 40% | decline |
| L03 score 699 | one point under the floor | decline |
| L04 boundaries | exactly 40% and exactly 700: "at most" and "or more" both pass | approve |
| L05 age at end | two rules pass; the third fails quietly: 54 + 7 = 61 | decline |
| L06 age 60 | 50 + 10 = 60, and "at most 60" passes | approve |
| L07 existing EMIs | the new EMI alone is 13% of income; with the existing EMIs it is 43.3% | decline |
| L08 two fail | score 640 **and** age at end 63; a model may stop at the first | decline |
| L09 large numbers | the numbers look big, but the ratio is 38% and age at end 53 | approve |
| L10 months | 48 months is 4 years, so 56 + 4 = exactly 60 | approve |

Five approvals and five declines, on purpose: a model that always says the same word scores exactly 5 of 10. Anything near 5/10 is no better than a coin.

**Analogy:** two loan officers get the same ten files. One must answer in one word. The other must write the calculation on each file first. You check both against the answer key and against the clock.

**Predict first**: which cases do you expect the *plain* version to get wrong? Write their ids in `predict_wrong` before you run anything.

### What the next cell does (the policy and the ten applicants)
1. Prints `LOAN_POLICY`, the one-line policy every case carries.
2. **`for c in LOAN_CASES:`** loops over the ten `Problem` objects. For each one, `c.text.splitlines()[1]` takes the second line of its text (line 0 is the policy, line 1 the applicant), and **`f"{c.id:<20}..."`** pads the id to 20 characters so the rows line up.
3. **`predict_wrong = []`** is where you type your prediction, as a list of id strings. No model call.

In [4]:
print(LOAN_POLICY + "\n")
for c in LOAN_CASES:
    print(f"{c.id:<20}{c.text.splitlines()[1]}")
predict_wrong = []     # YOUR TURN: e.g. ["L05-age-at-end", "L10-months"]

Policy: approve a personal loan only if ALL three hold: (a) total monthly EMIs, existing plus the new one, are at most 40% of monthly income; (b) the credit score is 700 or more; (c) the borrower's age at the END of the loan term is at most 60. Otherwise decline.

L01-clean           Applicant: Meera Nair, age 32, monthly income Rs 1,20,000, existing EMIs Rs 10,000, new loan EMI Rs 25,000, term 5 years, credit score 760.
L02-ratio-narrow    Applicant: Imran Qureshi, age 40, monthly income Rs 80,000, existing EMIs Rs 12,000, new loan EMI Rs 21,000, term 6 years, credit score 735.
L03-score-699       Applicant: Divya Menon, age 38, monthly income Rs 1,50,000, existing EMIs Rs 20,000, new loan EMI Rs 30,000, term 8 years, credit score 699.
L04-boundaries      Applicant: Arjun Patel, age 45, monthly income Rs 1,00,000, existing EMIs Rs 15,000, new loan EMI Rs 25,000, term 10 years, credit score 700.
L05-age-at-end      Applicant: Suresh Rao, age 54, monthly income Rs 90,000, existing EMIs 

### What the next cell does (the twenty calls)
1. If `ready`, loops over the ten cases and calls **`run(c, "plain")`** and **`run(c, "cot")`** for each: 20 calls, a few hundred to a few thousand tokens each.
2. **`case_table(results, LOAN_CASES, ("plain", "cot"))`** prints one row per case: the right answer, then what each strategy said with `ok` or `WRONG`.
3. **`strategy_summary(rows(strategies=("plain", "cot")), MODEL)`** prints one row per strategy: how many right, average tokens, average seconds, and the cost of 1,000 such questions.
4. **`got_wrong = [a.problem_id for a in rows(strategies=("plain",)) if not a.correct]`** is a *list comprehension*: "the id of each plain attempt, **if** it was not correct". It is printed under your prediction.
5. **`meter.report()`** shows the running total. Expect pauses for the rate limit here: twenty calls is the biggest block of the day.

In [5]:
if ready:
    for c in LOAN_CASES:
        run(c, "plain")
        run(c, "cot")
    print(case_table(results, LOAN_CASES, ("plain", "cot")))
    print()
    print(strategy_summary(rows(strategies=("plain", "cot")), MODEL))
    got_wrong = [a.problem_id for a in rows(strategies=("plain",)) if not a.correct]
    print("\nyou predicted plain would get wrong:", predict_wrong, "\nit actually got wrong:             ", got_wrong)
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

case                right answer  plain             cot               
----------------------------------------------------------------------
L01-clean           approve       approve ok        approve ok        
L02-ratio-narrow    decline       decline ok        decline ok        
L03-score-699       decline       decline ok        decline ok        
L04-boundaries      approve       approve ok        approve ok        
L05-age-at-end      decline       decline ok        decline ok        
L06-age-60          approve       approve ok        approve ok        
L07-existing-emis   decline       decline ok        decline ok        
L08-two-fail        decline       decline ok        decline ok        
L09-large-numbers   approve       approve ok        approve ok        
L10-months          approve       approve ok        approve ok        

strategy     right  avg tokens  avg secs  avg $/1000 q
------------------------------------------------------
plain        10/10         476       

### What you should see
A table, a summary, and your prediction next to the result. Shaped like this (an illustration, not a real run):
```
case                right answer  plain             cot
----------------------------------------------------------------------
L01-clean           approve       approve ok        approve ok
L02-ratio-narrow    decline       decline ok        decline ok
...
L05-age-at-end      decline       approve WRONG     decline ok
...
strategy     right  avg tokens  avg secs  avg $/1000 q
------------------------------------------------------
plain         8/10         520       1.9         $0.11
cot          10/10         960       4.2         $0.23
```
**How to read it:** first the `right` column of the summary (accuracy), then `avg tokens` and `avg secs` (cost and delay), then `$/1000 q` (money at scale). The interesting question is the **ratio**: how many extra right answers did step by step buy, and at how many times the tokens?

**Your numbers will be different from your neighbour's and from the instructor's.** Here are the possible outcomes, and each one teaches something:

| Outcome | What it means |
|---|---|
| cot gets more right than plain | the extra steps caught a trap that a one-word answer jumped over; now check the price |
| both get 10/10 | the reasoning model already did the work in its hidden thinking. That is a **finding, not a failure**: the visible working bought nothing on these cases, and cost extra |
| cot gets fewer right than plain | possible: written-out working can talk itself into a mistake. Read the reply (`results[("L05-age-at-end", "cot")].text`) to see where |
| a cell shows `none WRONG` | no readable `FINAL:` line, often because the reply was cut off at the token limit |

The `$/1000 q` column uses the price table in `reasoning_utils.py` for `openai/gpt-oss-20b` (indicative, October 2026). For any other model it shows a dash: no made-up price.

### Under the hood: rebuild the answer key
The whole lab depends on the answer key being right. If the key were wrong, every "WRONG" in your table would be a lie. So here is the decision rebuilt in a few lines, straight from the applicant's text, and checked against the key for all ten cases.

One detail matters: the ratio test uses **whole numbers**, `emis * 100 <= 40 * income`, not `emis / income <= 0.40`. Computers store most decimals approximately (the cell prints `0.1 * 3` to show it), and a rule like "at most 40%" must be exact when a case sits exactly on 40% (L04). Multiplying first keeps everything in whole numbers, so there is nothing to round.

**What the next cell does:**
1. `number_after(words, line)` uses a regular expression to find the number written after some words, for example `monthly income Rs 1,20,000`, removes the commas (Indian digit grouping) and returns `120000`.
2. `my_decision(case)` reads the applicant line, adds the two EMIs, converts a term in months to years (`48 // 12 = 4`; `//` is whole-number division), and applies the three rules. It returns the decision and two numbers to show.
3. Prints one row per case: the ratio, the score, the age at the end, your decision, the key, and `same` or `DIFFERENT`.

In [6]:
import re

def number_after(words, line):
    # the whole number written after `words`, e.g. 'monthly income Rs 1,20,000' -> 120000
    return int(re.search(words + r" (?:Rs )?([\d,]+)", line).group(1).replace(",", ""))

def my_decision(case):
    line = case.text.splitlines()[1]                              # line 0 is the policy, line 1 the applicant
    age, income = number_after("age", line), number_after("monthly income", line)
    emis = number_after("existing EMIs", line) + number_after("new loan EMI", line)
    score = number_after("credit score", line)
    n, unit = re.search(r"term (\d+) (years|months)", line).groups()
    years = int(n) if unit == "years" else int(n) // 12           # 48 months -> 4 years
    ratio_ok = emis * 100 <= 40 * income                           # whole numbers: exactly 40% passes, nothing to round
    decision = "approve" if ratio_ok and score >= 700 and age + years <= 60 else "decline"
    return decision, 100 * emis / income, age + years

print("why whole numbers?  0.1 * 3 =", 0.1 * 3, "\n")
print(f"{'case':<20}{'EMIs/income':>12}{'score':>7}{'age at end':>12}   {'mine':<9}{'key':<9}")
for c in LOAN_CASES:
    mine, ratio, age_end = my_decision(c)
    score = number_after("credit score", c.text.splitlines()[1])
    print(f"{c.id:<20}{ratio:>11.2f}%{score:>7}{age_end:>12}   {mine:<9}{c.truth:<9}{'same' if mine == c.truth else 'DIFFERENT'}")

why whole numbers?  0.1 * 3 = 0.30000000000000004 

case                 EMIs/income  score  age at end   mine     key      
L01-clean                 29.17%    760          37   approve  approve  same
L02-ratio-narrow          41.25%    735          46   decline  decline  same
L03-score-699             33.33%    699          46   decline  decline  same
L04-boundaries            40.00%    700          55   approve  approve  same
L05-age-at-end            39.44%    731          61   decline  decline  same
L06-age-60                27.27%    745          60   approve  approve  same
L07-existing-emis         43.33%    780          35   decline  decline  same
L08-two-fail              21.43%    640          63   decline  decline  same
L09-large-numbers         38.00%    705          53   approve  approve  same
L10-months                32.22%    712          60   approve  approve  same


**What you should see:** `0.1 * 3 = 0.30000000000000004` on the first line, then ten rows that all end in `same`. Look at the columns that create the traps: L02 at 41.25%, L04 at exactly 40.00% with score 700, L07 at 43.33%, and the age column at 61 (L05), 60 (L06 and L10) and 63 (L08).

**Python notes:** `re.search(...).group(1)` returns the part of the match inside the first pair of brackets. `n, unit = ....groups()` unpacks two matched parts into two names. `{ratio:>11.2f}` prints a number right-aligned in 11 characters with 2 decimals.

**The take-away:** the model is judged against rules **you** can write in ten lines. Whenever a decision can be written as rules like these, ask yourself whether the model should make it at all, or only explain it. Session 20 (function calling) and Session 34 (agents) build on exactly that question.

### Check yourself
*Question 2 — Put your predictions next to the table. Did step-by-step change the number of right answers? Did it change the cost? Which of the two moved more, and what does that say about when to pay for it?*

> **Common mistake:** concluding "step by step always helps" (or "never helps") from one run on ten cases. It depends on the model and on the kind of question. Ten cases is a small sample: one case is 10 percentage points.

## 4. Step back

### What and why
**Step-back prompting** asks for the *principles* first, in a call that is not allowed to solve anything, and only then solves with those principles in front of it. Two calls instead of one. We try it on the three cases that gave the most trouble above (or, if nothing was wrong, the three that cost the most thinking). Spending the extra call only on the hard cases is itself a lesson: you do not pay for the expensive method everywhere.

### The idea: the pilot's checklist
```
 call 1                                        call 2
 ────────────────────────────────────         ──────────────────────────────────────────────
 the case                                      the case
 + "Do NOT solve this yet. Step back:    ──►   + "Rules and formulas to apply:" + call 1's bullets
   list, in at most five short bullet          + "Think step by step and show each calculation.
   points, the general rules, formulas           Then end with ... FINAL: <answer>"
   or traps that apply to this kind
   of problem."                                → read the FINAL line, compare with the key
 → up to five bullets (the principles)
```
**Analogy:** a pilot reads the checklist for *this type of flight* before flying *this flight*. The checklist is written without looking at today's weather; that is the point. Here, call 1 is forbidden to solve, so it cannot jump to an answer and then justify it.

When does it help? When the trap is **knowing which rule applies**, or recalling a formula. When does it not? When the rules are already written in the prompt, as our loan policy is. Then call 1 may only restate the policy, and you paid for two calls. Read what your model wrote and judge it honestly.

### What the next cell does
1. **`trouble = pick_trouble(rows(strategies=("plain", "cot")), LOAN_CASES, 3)`** picks the three cases answered wrongly most often in section 3. Ties (including "nothing wrong") go to the cases that used the most tokens. This line runs even without a key; then it simply returns the first three cases.
2. Prints the ids it chose.
3. If `ready`, runs **`run(c, "stepback")`** on each: 2 calls per case, 6 in all.
4. Prints what the model wrote in call 1 for the first case. It is stored in the attempt's **`notes`** dictionary under `"principles"`.
5. Prints the case table with three columns (plain, cot, stepback) and the summary for these three cases only.

In [7]:
trouble = pick_trouble(rows(strategies=("plain", "cot")), LOAN_CASES, 3)
print("step back on:", [c.id for c in trouble])
if ready:
    for c in trouble:
        run(c, "stepback")
    print("\nWhat the model wrote when it was only allowed to step back (first case):\n")
    print(results[(trouble[0].id, "stepback")].notes["principles"])
    print("\n" + case_table(results, trouble, ("plain", "cot", "stepback")))
    print()
    print(strategy_summary(rows(trouble, ("plain", "cot", "stepback")), MODEL))
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

step back on: ['L03-score-699', 'L05-age-at-end', 'L08-two-fail']

What the model wrote when it was only allowed to step back (first case):

- **EMI‑to‑income check**:  
  - Compute total monthly EMIs = existing EMI + new loan EMI.  
  - Compute 40 % of monthly income.  
  - Condition (a) holds if total EMIs ≤ 40 % of income.

- **Credit‑score threshold**:  
  - Condition (b) holds if credit score ≥ 700.

- **Age‑at‑term check**:  
  - Age at loan end = current age + loan term (in years).  
  - Condition (c) holds if this age ≤ 60.

- **All‑or‑nothing rule**:  
  - Approval requires *simultaneous* satisfaction of (a), (b), and (c).  
  - Failure of any single condition triggers a decline.

- **Common pitfalls**:  
  - Forgetting to add existing EMIs to the new EMI.  
  - Mis‑calculating 40 % of income (use decimal 0.40).  
  - Ignoring that the age check uses the *end* of the term, not the current age.

case                right answer  plain             cot               stepback     

### What you should see
`step back on: [...]` with three ids, then a few bullet points, then a three-column table and a three-row summary. An illustration of the principles, not a real run:
```
- Add existing and new EMIs before comparing with 40% of income.
- "At most" and "or more" include the boundary value.
- Age at the END of the term = current age + term in years; convert months to years.
```
**How to read it:** ask two questions. *Were the bullets useful, or just the policy in other words?* And *did the stepback column fix anything that cot had not already fixed?* Then look at `avg tokens`: step back is usually the most expensive single-answer method, because it makes two calls and the second prompt is longer.

| If you see... | It means... | Do this |
|---|---|---|
| `step back on: ['L01-clean', 'L02-ratio-narrow', 'L03-score-699']` without a key | `pick_trouble` had no results, so it took the first three | normal with no key |
| the bullets restate the policy almost word for word | the rules were already in the prompt | a real finding: step back added cost, not knowledge, for this task |
| stepback is WRONG where cot was ok | the principles pushed the model the wrong way | read both replies; one case is not a trend |

### Check yourself
*Question 3 — Read the principles the model wrote. Were they useful, or did they just restate the policy you had already given it? When would you expect step-back to beat step-by-step, and when not?*

> **Common mistake:** letting call 1 solve the problem "to save a call". Then call 2 is just a second chain of thought, and you paid twice. The rule "do NOT solve this yet" is the technique.

## 5. Take a vote (self-consistency)

### What and why
Ask the same step-by-step question several times at a temperature above zero, so the answers can differ, and take the **most common answer**. It is how you buy reliability with money: five times the calls for one answer. We spend it on one case only, the one that gave the most trouble.

### The idea: five underwriters, one committee
```
             ┌─► sample 0 (temperature 0.8) ─► FINAL: decline ─┐
             ├─► sample 1                    ─► FINAL: approve ─┤
 the case  ──┼─► sample 2                    ─► FINAL: decline ─┼─►  count: decline 3, approve 2
 + "think    ├─► sample 3                    ─► FINAL: decline ─┤    majority: decline (agreement 60%)
 step by     └─► sample 4                    ─► FINAL: approve ─┘
 step"                          (illustration: these votes are made up)
```
**Analogy:** five junior underwriters review the same file *independently*, and the committee goes with the majority. "Independently" matters: if they copy each other, you have one opinion five times. For a model, independence comes from **temperature above zero** (the helper uses 0.8). The `sample` number 0 to 4 also goes into the DiskCache key, so the memory keeps five different answers instead of returning the first one five times.

What voting can and cannot fix:
- It fixes **random slips**: a model that is right most of the time but sometimes trips.
- It cannot fix a **systematic error**: if the model applies a wrong rule, all five apply it, and they agree perfectly on the wrong answer.
- It needs **one checkable answer** to count. Voting on essays only averages opinions.

### What the next cell does
1. **`VOTES = 5`** sets the number of samples.
2. If `ready`, **`hardest = pick_hardest(...)`** picks the single most troublesome of the three step-back cases, and prints it.
3. **`a = run(hardest, "vote", n=VOTES)`** makes 5 calls. The `n=VOTES` travels through `run`'s `**kw` into the helper. The attempt is stored under the strategy name `"vote x5"`.
4. Prints the votes (a dictionary from answer to count), the **agreement** as a percentage (`:.0%` turns `0.6` into `60%`), how many single samples were right, and whether the majority was right.
5. **`single = results.get((hardest.id, "cot"))`** looks up the one-call step-by-step result for the same case. **`dict.get`** returns `None` instead of raising an error when the key is missing, so the cost comparison only prints if that result exists.

In [8]:
VOTES = 5
if ready:
    hardest = pick_hardest(rows(trouble, ("plain", "cot", "stepback")), trouble)
    print("voting on:", hardest.id, "-", hardest.title)
    a = run(hardest, "vote", n=VOTES)
    print("\nvotes:", a.notes["votes"], "| agreement:", f"{a.notes['agreement']:.0%}")
    print(f"single samples right: {a.notes['single_sample_correct']}/{VOTES}     the majority is right: {a.correct}")
    single = results.get((hardest.id, "cot"))
    if single:
        print(f"cost: {a.calls} calls, {a.tokens:,} tokens, {a.seconds:.0f}s   vs one step-by-step call: {single.tokens:,} tokens, {single.seconds:.0f}s")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

voting on: L08-two-fail - Two rules fail at once

votes: {'decline': 5} | agreement: 100%
single samples right: 5/5     the majority is right: True
cost: 5 calls, 3,874 tokens, 5s   vs one step-by-step call: 782 tokens, 3s

spent so far: 0 calls (32 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


### What you should see
Shaped like this (an illustration, not a real run):
```
voting on: L05-age-at-end - Age at the end of the term

votes: {'decline': 4, 'approve': 1} | agreement: 80%
single samples right: 4/5     the majority is right: True
cost: 5 calls, 4,800 tokens, 21s   vs one step-by-step call: 960 tokens, 4s
```
**How to read it:** the *agreement* is a free confidence signal: 100% means the model was sure (not necessarily right); 60% means the case is hard or the rule is ambiguous. Then compare `single samples right` with `the majority is right`: the vote helps only when single samples are right more often than not. Last, the cost line: about five times one call.

| If you see... | It means... |
|---|---|
| 5/5 agree and the majority is right | the case was easy for this model; the vote bought nothing but cost |
| a split, and the majority is right | the vote rescued a case that single samples sometimes got wrong: what you paid for |
| 5/5 agree and the majority is WRONG | a systematic error: more votes will never fix it; fix the prompt or the rule |

### Under the hood: count the votes yourself
The vote is just counting. Here is `majority_vote` rebuilt with a plain dictionary, checked against the real helper on six lists of votes, including a tie, missing answers (`None`, a reply with no readable FINAL line) and an empty list.

**What the next cell does:**
1. Imports the real `majority_vote`.
2. Defines `my_majority_vote(keys)`: skip `None`; count each answer with **`counts[k] = counts.get(k, 0) + 1`** (`dict.get(k, 0)` gives 0 the first time an answer is seen); find the highest count; return the *first* answer with that count, the count, and the number of valid votes. Python dictionaries remember the order in which keys were added, which is why "first seen wins a tie" works.
3. Prints both results for each list, `same` or `DIFFERENT`, and the agreement.

In [9]:
from reasoning_utils import majority_vote                        # the real helper, to compare against

def my_majority_vote(keys):
    counts = {}
    for k in keys:
        if k is not None:                                          # an unreadable reply does not vote
            counts[k] = counts.get(k, 0) + 1
    if not counts:
        return None, 0, 0
    best = max(counts.values())
    winner = next(k for k in counts if counts[k] == best)          # dicts keep insertion order: first seen wins a tie
    return winner, best, sum(counts.values())

ballots = [["decline", "approve", "decline", "decline", "approve"],
           ["approve", "approve", "approve", "approve", "approve"],     # perfect agreement: confident, not necessarily right
           ["approve", "decline", "approve", "decline"],                # a tie: the answer seen first wins
           ["decline", None, None, "approve", "decline"],
           [None, None],
           []]
for b in ballots:
    mine, real = my_majority_vote(b), majority_vote(b)
    winner, votes, valid = real
    agreement = f"{votes / valid:.0%}" if valid else "-"
    print(f"{'same' if mine == real else 'DIFFERENT':<10}{str(b):<58}-> {str(winner):<8} {votes}/{valid}  agreement {agreement}")

same      ['decline', 'approve', 'decline', 'decline', 'approve']   -> decline  3/5  agreement 60%
same      ['approve', 'approve', 'approve', 'approve', 'approve']   -> approve  5/5  agreement 100%
same      ['approve', 'decline', 'approve', 'decline']              -> approve  2/4  agreement 50%
same      ['decline', None, None, 'approve', 'decline']             -> decline  2/3  agreement 67%
same      [None, None]                                              -> None     0/0  agreement -
same      []                                                        -> None     0/0  agreement -


**What you should see:** six rows starting with `same`. The second ballot shows 100% agreement on `approve`: if the key said `decline`, that would be the "wrong rule" case, perfectly agreed and perfectly wrong. The tie goes to `approve` because it was seen first; the ballot with two `None`s has 3 valid votes, not 5.

**Python notes:** `next(k for k in counts if ...)` takes the first item of a *generator expression* (a list comprehension in round brackets that produces items one at a time). `str(b):<58` turns the list into text and pads it.

### Check yourself
*Question 4 — The tables give a cost per 1,000 questions for each strategy. Multiply it for 200,000 loan decisions a month. What would a 5x vote add to the monthly bill, and what size of accuracy gain would justify it? For which kinds of question would you never vote?*

<details><summary><b>▸ Read more: working out the monthly bill</b></summary>

A worked example with **made-up** numbers, so you can do the same with yours. Suppose your summary says one step-by-step call costs about $0.23 per 1,000 questions. At 200,000 loan decisions a month that is 200 × $0.23 = $46 a month. A 5x vote is five such calls per decision: about $230 a month, so the vote *adds* about $184. Is that worth it? Compare it with the cost of one wrong decision: a wrongly approved loan that defaults, or a wrongly declined good customer. If the vote prevents even a handful of wrong decisions a month, it may pay for itself; if your single calls are already right on every case you tested, it buys nothing. Also count **seconds**: five calls in a row take five times as long, which matters in a live app.
</details>

> **Common mistake:** voting at temperature 0. The five samples are then (nearly) identical, the agreement is always 100%, and you paid five times for one opinion.

## 6. The dial: reasoning effort

### What and why
This model has a dial: `reasoning_effort` can be `"low"`, `"medium"` or `"high"`. The same plain question about the same case, only the dial changes. (Some models and providers do not have a dial; the cell says so instead of failing.) The dial is the cheapest experiment of the day: two calls tell you whether this model needs a deep review for this kind of case, or whether a skim is enough.

### The idea: how long the underwriter spends on the file
```
  effort = "low"      ───►  skim             fewer thinking tokens, faster, cheaper
  effort = "medium"   ───►  normal review
  effort = "high"     ───►  deep review      more thinking tokens, slower, dearer
                             (whether "deeper" is also "more right" is what you measure)
```
**Analogy:** the same loan file, given to an underwriter with five minutes or with an hour. The hour costs more. Whether it changes the decision depends on the file.

How it is sent: the helper adds `extra_body={"reasoning_effort": effort}` to the request. `extra_body` is the OpenAI client's way of passing a setting the client library does not know about. That is why a provider or model without the dial can reject it, and why the cell catches the error.

### What the next cell does
1. **`dial = []`** collects the two attempts.
2. If `ready`, loops over `("low", "high")`. For each, **`lab.variant(effort=effort)`** makes a copy of the lab with only the effort changed (same cache, same meter), and `run_strategy(...)` asks the plain question about `hardest` (from section 5, or L05 by default).
3. **`a.strategy = f"plain/{effort}"`** renames the attempt so the table shows `plain/low` and `plain/high`. These attempts go into `dial`, **not** into `results`, so they do not appear in your accuracy table.
4. **`try: ... except Exception as e:`** catches a rejection. The message is shortened with **`str(e)[:120]`** (the first 120 characters). **`{effort!r}`** prints the value with quotes, as `'low'`.
5. If at least one worked, prints **`comparison_table(dial, MODEL)`**: answer, result, tokens, **thinking** tokens, seconds and $ per 1,000 for each setting.

In [10]:
dial = []
if ready:
    for effort in ("low", "high"):
        try:
            a = run_strategy(lab.variant(effort=effort), hardest, "plain")
            a.strategy = f"plain/{effort}"
            dial.append(a)
        except Exception as e:
            print(f"effort={effort!r} was not accepted by this model or provider: {str(e)[:120]}")
    if dial:
        print(comparison_table(dial, MODEL))
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
L08-two-fail          plain/low    decline  ok         1     344       81    1.5     $0.05
L08-two-fail          plain/high   decline  ok         1     598      335    1.2     $0.12

spent so far: 0 calls (34 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


### What you should see
A two-row table, shaped like this (an illustration, not a real run):
```
problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
L05-age-at-end        plain/low    approve  WRONG      1     310       72    0.8     $0.04
L05-age-at-end        plain/high   decline  ok         1   1,120      720    3.9     $0.29
```
**How to read it:** look at the `thinking` column first. That is what the dial moves. Then `result`: did the extra thinking change the answer? Then `secs` and `$/1000 q`: what did it cost?

| If you see... | It means... | Do this |
|---|---|---|
| `effort='low' was not accepted by this model or provider: ...` | no dial here | a normal finding; say "this model has no dial" and move on |
| low and high give the same answer | for this case, the skim was enough | the extra thinking bought nothing here; test more cases before deciding |
| thinking shows `-` | the provider reported no reasoning tokens | compare total tokens and seconds instead |

### Check yourself
*Question 5 — If low and high gave the same answer, what did the extra thinking buy you? If they differed, which would you ship, and how would you know from one case that it was not luck?*

> **Common mistake:** setting `"high"` everywhere "to be safe". It is paid on every call. Decide per kind of question, from measurements on more than one case.

## 7. When thinking does not help

### What and why
Back to Session 14's complaint classifier: eight messages, the bank's own convention (UPI problems are `account`, servicing requests are `other`). Three ways: the plain one-word version, the same with *thinking*, and the Session 14 version with **four worked examples**. Remember what the examples were for.

This section is the **control experiment**. Sections 3 to 6 asked "how much does thinking help?". This one asks "is there a kind of question where it *cannot* help?". The answer protects you from the most expensive mistake of the day: adding "think step by step" everywhere.

### The idea: thinking works with the rules it has; it cannot invent the missing one
```
 "UPI payment failed, money debited"
          │
          ├── zero-shot, one word ─────► the model's general idea: "UPI = payments = other?" ─► maybe wrong
          ├── zero-shot + thinking ────► the same idea, argued in more detail ───────────────► maybe still wrong
          └── four examples, one word ─► an example shows: transfer pending → account ─► house rule
                                         (the bank's convention, taught by showing)        now in the prompt
```
**Analogy:** a new employee who was never told the bank's filing rules. Asking her to "think harder" produces a careful, well-argued, wrong folder. Showing her four filed examples teaches the rule in a minute.

The thinking version is built by `classify_cot_spec()`: it takes the Session 14 instruction and replaces *"Reply with that single word only."* with *"Think step by step in a few lines, then end with one last line in the form FINAL: &lt;category&gt;."* Everything else is identical, so the comparison is fair.

### What the next cell does
1. If `ready`, builds a dictionary **`control`** with three entries. Each value is the result of **`score_classifier_lab(lab, spec, mode=...)`**, which runs one prompt over the eight labelled messages (8 calls each, 24 in all) and returns right/8, tokens, seconds and the wrong ones.
   - `classify_spec(0)`: the Session 14 instruction, zero examples, read as one word (`mode="plain"`).
   - `classify_cot_spec()`: the same, asked to think, read from the `FINAL:` line (`mode="cot"`).
   - `classify_spec(4)`: the Session 14 instruction with four worked examples, one word.
2. Prints a small table with **`f"{'variant':<26}{'right':>7}..."`**: `<26` pads left-aligned to 26 characters, `>7` right-aligns in 7. `{r['tokens']:>9,}` adds thousands separators.
3. **`for name, r in control.items():`** loops over the dictionary's pairs, and **`r["wrong"][:2]`** takes at most the first two mistakes of each variant. **`for text, truth, got in ...`** unpacks each mistake into three names. **`{text[:52]!r}`** shows the first 52 characters with quotes.

In [11]:
if ready:
    control = {
        "zero-shot, one word": score_classifier_lab(lab, classify_spec(0), mode="plain"),
        "zero-shot + thinking": score_classifier_lab(lab, classify_cot_spec(), mode="cot"),
        "four examples, one word": score_classifier_lab(lab, classify_spec(4), mode="plain"),
    }
    print(f"{'variant':<26}{'right':>7}{'tokens':>9}{'secs':>7}")
    for name, r in control.items():
        print(f"{name:<26}{r['accuracy']:>4}/{r['n']:<2}{r['tokens']:>9,}{r['seconds']:>7.1f}")
    print()
    for name, r in control.items():
        for text, truth, got in r["wrong"][:2]:
            print(f"  [{name}] wrong: {text[:52]!r} -> {got!r}, the bank says {truth!r}")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

variant                     right   tokens   secs
zero-shot, one word          6/8     1,539    4.5
zero-shot + thinking         6/8     1,938    4.5
four examples, one word      6/8     3,195    4.9

  [zero-shot, one word] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [zero-shot, one word] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'
  [zero-shot + thinking] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [zero-shot + thinking] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'
  [four examples, one word] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [four examples, one word] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'

spent so far: 0 calls (58 answered from memory), 0 token

### What you should see
A three-row table and up to two mistakes per variant. Shaped like this (an illustration, not a real run):
```
variant                     right   tokens   secs
zero-shot, one word          6/8     1,360    3.1
zero-shot + thinking         6/8     2,160    6.0
four examples, one word      8/8     1,900    3.3

  [zero-shot, one word] wrong: 'UPI payment of Rs 4,500 failed but the money was debi'
      -> 'other', the bank says 'account'      (the real output prints this on one line)
```
**How to read it:** compare *right* down the column, then *tokens*. The usual pattern is that thinking costs more and fixes nothing that depends on the house convention, while examples fix it. If thinking fixed something, read which message: it was probably a case general knowledge could solve, not the house rule.

| If you see... | It means... |
|---|---|
| all three 8/8 | this model already guessed the convention on these eight; examples are still the reliable way to *state* it |
| thinking fixes a label | the mistake did not depend on the house rule; check which message |
| `-> None` in a mistake | the reply had no readable label (for thinking: no `FINAL:` line); counted as wrong |

### Check yourself
*Question 6 — Which change fixed the wrong labels: thinking, or examples? Why can no amount of thinking recover a rule the model was never told? What does that say about choosing between the two?*

> **Common mistake:** treating "think step by step" as a general quality booster. It helps when the rules are present and the steps chain. When a rule is missing, it adds tokens and confidence, not correctness.

## 8. A different kind of model: your local Llama

### What and why
Session 12's small local model does **not** think before it answers. Five of the loan cases, plain and step by step, now on your laptop. No tokens are billed, so this section does not touch your daily limit; it does take a few minutes on a laptop CPU, so start it and carry on reading. If Ollama is not running, the cell says so and moves on.

Why bother? Because the advice "always add *think step by step*" was born on models like this one. Comparing a model that thinks first with one that does not shows you *why* the same sentence can matter a lot on one and little on the other.

### The idea: where does the thinking happen?
```
 hosted reasoning model:   question ─► [hidden thinking, billed] ─► answer
                           "think step by step" asks for what it has already begun

 local small model:        question ─────────────────────────────► answer   (no thinking first)
                           "think step by step" is the ONLY place its working can happen
```
**Analogy:** an experienced officer who always works the file in his head, versus a trainee who only works it if you hand her a form with boxes for each step. The form changes the trainee's results far more.

### What the next cell does
1. **`LOCAL_IDS = ("L02", "L05", "L07", "L09", "L10")`** is a tuple of id prefixes. **`c.id.startswith(LOCAL_IDS)`** accepts a whole tuple and is true if the id starts with *any* of them. The list comprehension keeps those five cases.
2. **`try:`** makes an Ollama client with a short timeout (`with_options(timeout=5, max_retries=0)`, so a missing Ollama fails in seconds, not minutes) and lists the installed models. **`pick_local_chat_model`** skips embedding models, which cannot answer questions.
3. **`except Exception as e:`** prints "Ollama is not reachable" instead of crashing.
4. If a model was found, builds a **separate** `Lab` for it, with `max_tokens=1500`, the same DiskCache, and a new `Meter()` (so the hosted meter is not mixed with local calls).
5. Runs plain and cot on the five cases (10 local calls) and prints **`comparison_table(local_results)`** and the summary. No model name is passed for prices, so the `$` columns show a dash: a local model costs electricity, not tokens.

In [12]:
LOCAL_IDS = ("L02", "L05", "L07", "L09", "L10")
local_cases = [c for c in LOAN_CASES if c.id.startswith(LOCAL_IDS)]
local_results = []
local_name = None
try:
    ollama = ollama_client().with_options(timeout=5, max_retries=0)
    local_name = pick_local_chat_model([m.id for m in ollama.models.list().data])
except Exception as e:
    print("Ollama is not reachable:", str(e)[:100])
if local_name:
    print("local model:", local_name)
    local = Lab(ollama_client(), local_name, max_tokens=1500, cache=cache, meter=Meter())
    for c in local_cases:
        for strategy in ("plain", "cot"):
            local_results.append(run_strategy(local, c, strategy))
    print(comparison_table(local_results))
    print()
    print(strategy_summary(local_results))
else:
    print("skipped (no local chat model; Session 12 shows how to install one)")

Ollama is not reachable: Connection error.
skipped (no local chat model; Session 12 shows how to install one)


### What you should see
With Ollama running: `local model: llama3.2:3b` (or whichever you installed), a ten-row table and a two-row summary. The `thinking` column shows `-`, because this model does not think first. Without Ollama: `Ollama is not reachable: ...` and `skipped (...)`. Both are fine.

| If you see... | It means... | Do this |
|---|---|---|
| `Ollama is not reachable: Connection error.` | Ollama is not running | optional: start Ollama (Session 12), re-run; or skip this section |
| `skipped (no local chat model ...)` | only an embedding model is installed | `ollama pull` a small chat model (Session 12) |
| very long run time | a laptop CPU is slow on long replies | let it run while you read; re-runs are free (DiskCache) |

### Check yourself
*Question 7 — Compare the local model's plain and step-by-step rows with the hosted model's on the same cases. Which model gained more from step-by-step, and why? What does that tell you about advice such as 'always add think step by step'?*

> **Common mistake:** comparing the local model's *seconds* with the hosted model's and concluding the local one is "worse". It runs on your laptop's CPU. Compare accuracy and the gain from step by step, not speed.

## 9. Other domains (optional)

### What and why
The same four ways of asking, on an insurance claim, a retail order and a factory defect puzzle, then (stretch) healthcare dosing with an invented formulary (not medical advice), telecom proration and an EMI formula. The EMI is a calculation a calculator does exactly: in Session 20 you give the model a tool for it. Leave `RUN_OTHER = False` to save tokens.

### The idea: different traps, same method
| Problem | Domain | The trap | Answer key |
|---|---|---|---|
| R1 retail order | Retail | rules chain: the coupon test uses the discounted amount; free shipping is judged before GST | Rs 5,779 (5,778.99) |
| R2 insurance claim | Insurance | order matters: deductible before co-pay gives 1,26,000; co-pay first gives 1,25,000 | Rs 1,26,000 |
| R4 defect source | Manufacturing | the tempting answer is 60% (machine A's share); the right one weighs share by defect rate | 37.5% |
| S1 dosage | Healthcare | per-dose mg first (90 mg), then mL; the 200 mg cap is a check, not a number to use | 3.6 mL |
| S2 proration | Telecom | day 12 to day 30 inclusive is 19 days, not 18 | Rs 253 (253.33) |
| S3 EMI | BFS | (1.01)^36 is about 1.4308; a slip moves the EMI by hundreds | Rs 16,607 |

R4 is the kind of question where **step back** can shine: the trap is *recalling the right formula* (Bayes' rule), not following written rules. S3 is the kind where no prompt is the right answer: a calculator is exact, cheap and never needs a vote.

### What the next cell does
1. **`RUN_OTHER = False`** is the switch. Set it to `True` only if you have tokens to spare.
2. If `ready and RUN_OTHER`, loops over **`OTHER + STRETCH`** (adding two lists makes one list of six problems) and runs plain and cot on each: 12 calls.
3. Prints a comparison table for these six problems only, and the meter.

In [13]:
RUN_OTHER = False
if ready and RUN_OTHER:
    for p in OTHER + STRETCH:
        run(p, "plain")
        run(p, "cot")
    print(comparison_table(rows(OTHER + STRETCH, ("plain", "cot")), MODEL))
    print()
    print(meter.report())
else:
    print("skipped (set RUN_OTHER = True to run; needs a key)")

skipped (set RUN_OTHER = True to run; needs a key)


### What you should see
With the switch off: `skipped (set RUN_OTHER = True to run; needs a key)`. With it on: a twelve-row table. For number answers, a small tolerance applies (for example, within 1 rupee for R1), so `5779` and `5778` both count as right.

### Check yourself
*Before you switch it on: for which of the six problems do you expect step by step to beat plain, and for which would you never pay for thinking at all? Write it in your decision table (section 12).*

> **Common mistake:** paying for votes on arithmetic. A vote makes a wrong sum less likely; a calculator makes it impossible. That is Session 20.

## 10. Reasoning behind a curtain

### What and why
In a real product the customer must see the **answer**, not the working. So ask for the working inside `<scratchpad>` tags and the real reply inside `<final>` tags, and let your *code* strip the scratchpad. Two rules follow: the acceptance checks run on the `<final>` part only, and the scratchpad is **never shown or stored where a customer can read it**, because private working can mention things the reply must not (Task 2's confidential score is the example).

Below, the same reference prompts as Session 14, version 1 against version 2 with the curtain, run twice each at a temperature above zero. `BASE` is where your own Session 14 prompts would go: copy the parts from your `library.md` into a `PromptSpec` to improve *your* prompt.

### The idea: the back of the envelope
```
 the model's reply
 ┊ <scratchpad>                                   ┊
 ┊   score 612 is below the cutoff 650 ...        ┊ ──► stripped by YOUR code: never shown, never in a
 ┊ </scratchpad>                                  ┊     customer-readable log; checked for leaks only
 ┊ <final>                                        ┊
 ┊   Hi Rohan, your application APP-20931 was     ┊ ──► the only part that is checked, shown and stored
 ┊   not approved. You can apply again after ...  ┊
 ┊ </final>                                       ┊
                          (an illustration, not a recorded output)
```
**Analogy:** you work out a difficult letter on the back of an envelope, then post the clean letter. Nobody posts the envelope.

Task 1 (the credit-committee summary, 4 checks) and Task 2 (the declined-loan SMS, 6 checks) are Session 14's tasks with their acceptance checks. Task 2 has one check that matters most: the SMS must **not** contain `612`, `650`, `risk score` or `cutoff`. The helper also looks for those four terms in the *scratchpad*, to show you where the secret appears.

### What the next cell does (what version 2 adds)
1. **`BASE = {TASK1.id: TASK1.reference, TASK2.id: TASK2.reference}`** is a dictionary from task id to a Session 14 `PromptSpec`. Replace the values with your own specs if you want to improve *your* prompts.
2. **`with_scratchpad(BASE[TASK2.id])`** returns a **copy** of Task 2's prompt with one extra rule (do your reasoning inside `<scratchpad>`, which is private) and a new output format (a scratchpad block, then `<final>...</final>` containing the original format). The original spec is not changed.
3. **`.to_messages(TASK2.user_input)[0]["content"]`** builds the messages and prints the first one, the system message, so you can see exactly what version 2 adds. No model call.

In [14]:
BASE = {TASK1.id: TASK1.reference, TASK2.id: TASK2.reference}      # YOUR TURN (optional): put your own v1 specs here
print("What version 2 adds to Task 2's standing rules:\n")
print(with_scratchpad(BASE[TASK2.id]).to_messages(TASK2.user_input)[0]["content"])

What version 2 adds to Task 2's standing rules:

You are a customer-communications writer at a retail bank.

Rules:
- Use at most 280 characters.
- Address the customer by first name.
- Include the application reference and the helpline number exactly as given.
- Mention the reapply period.
- Never reveal internal scores, cutoffs or the internal reason.
- Do your reasoning inside <scratchpad>...</scratchpad> first. The scratchpad is private working and is never shown to anyone.

Output format: A <scratchpad> block, then <final>...</final> containing exactly this: One plain-text SMS. No subject line, no markdown, no emojis.


### What the next cell does (v1 against v2)
1. **`RUNS = 2`** runs each version twice. `pass_rate` uses temperature 0.7 and sample numbers 0 and 1, so the two runs can differ, and both are cached.
2. For each task, **`pass_rate(lab, task, BASE[task.id], runs=RUNS)`** runs version 1 and counts the checks it passed. With **`hidden_reasoning=True`**, it runs version 2, splits each reply into scratchpad and final, and runs the checks **on the final part only**.
3. Stores both in **`compare[task.id]`** as a pair `(v1, v2)`; section 11 needs it to save version 2.
4. Prints average checks passed, how many runs passed every check, and average tokens per call, then **`verdict(v1, v2)`**: one sentence saying whether v2 is better, worse or the same, and at what token ratio.
5. If any scratchpad mentioned a confidential term, prints the `!!` warning. 2 tasks × 2 versions × 2 runs = 8 calls.

In [15]:
RUNS = 2
compare = {}
if ready:
    for task in (TASK1, TASK2):
        v1 = pass_rate(lab, task, BASE[task.id], runs=RUNS)
        v2 = pass_rate(lab, task, BASE[task.id], hidden_reasoning=True, runs=RUNS)
        compare[task.id] = (v1, v2)
        print(f"{task.id}")
        print(f"   v1: {v1['avg_checks']:.1f} of {v1['of']} checks, every check passed in {v1['all_passed']}/{RUNS} runs, {v1['avg_tokens']:,.0f} tokens a call")
        print(f"   v2: {v2['avg_checks']:.1f} of {v2['of']} checks, every check passed in {v2['all_passed']}/{RUNS} runs, {v2['avg_tokens']:,.0f} tokens a call")
        print("   ->", verdict(v1, v2))
        if v2["any_leak"]:
            print("   !! the scratchpad MENTIONS the confidential values. That is why it is never shown to the customer.")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

T1-credit-summary
   v1: 4.0 of 4 checks, every check passed in 2/2 runs, 1,018 tokens a call
   v2: 4.0 of 4 checks, every check passed in 2/2 runs, 1,354 tokens a call
   -> v1 and v2 pass the same number of checks (4.0 of 4): no reason to switch, at 1.3x the tokens per call.
T2-sms-declined
   v1: 6.0 of 6 checks, every check passed in 2/2 runs, 972 tokens a call
   v2: 6.0 of 6 checks, every check passed in 2/2 runs, 1,030 tokens a call
   -> v1 and v2 pass the same number of checks (6.0 of 6): no reason to switch, at 1.1x the tokens per call.

spent so far: 0 calls (66 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


### What you should see
The system message of Task 2 version 2 (ending with the new output format) from the first cell. From the second, shaped like this (an illustration, not a real run):
```
T1-credit-summary
   v1: 3.5 of 4 checks, every check passed in 1/2 runs, 390 tokens a call
   v2: 4.0 of 4 checks, every check passed in 2/2 runs, 610 tokens a call
   -> v2 passes more checks (4.0 vs 3.5 of 4), at 1.6x the tokens per call.
T2-sms-declined
   v1: 6.0 of 6 checks, every check passed in 2/2 runs, 370 tokens a call
   v2: 6.0 of 6 checks, every check passed in 2/2 runs, 560 tokens a call
   -> v1 and v2 pass the same number of checks (6.0 of 6): no reason to switch, at 1.5x the tokens per call.
   !! the scratchpad MENTIONS the confidential values. That is why it is never shown to the customer.
```
**How to read it:** the verdict line is the one you will copy, in numbers, into `WHY` in section 11. The `!!` line is the reason for the curtain: the working knew the secret even though the reply did not say it. If the `!!` line does not appear, the scratchpad happened not to mention the terms this time; the rule still stands.

| If you see... | It means... | Do this |
|---|---|---|
| `v2 passes FEWER checks ...: keep v1` | the curtain cost more and helped less | a valid result; you may still keep v2 for privacy, but say so honestly in `WHY` |
| v2 checks fail although the SMS looks fine | the reply was cut off before `</final>`, or had no tags | look at `compare[TASK2.id][1]["rows"][0]["final"]` |

### Check yourself
*Question 8 — Task 2's reply must never reveal the internal score. Did the scratchpad mention it? Who inside the bank may read a scratchpad, where would you store it, and for how long? Would you still keep the curtain if v2 gave no better check results?*

> **Common mistake:** running the acceptance checks on the whole reply. The scratchpad contains the score, so the "never reveal the score" check fails even when the customer would never see it, or, worse, someone "fixes" it by showing the customer the whole reply.

## 11. Your work repository: the accuracy table and prompt library v2

### What and why
Two things go into your **work** repository today.

1. **The accuracy table for the ten cases** (`experiments/s15_accuracy_table.md`), made from the results above. This is the session's checkpoint.
2. **Prompt library v2 for Tasks 1 and 2.** Same rule as Session 14: **never edit an old entry, add a new one.** Only save a v2 if it earned its place, and say why **in numbers** in `WHY` (for example `"4.0 vs 3.0 of 4 checks, 2.1x tokens, worth it for a credit decision"`).

The save cell stays off until you set `SAVE_V2 = True`, fill in both reasons, and have run sections 3 and 10.

### The idea: results become evidence when they are committed
```
 results (sections 3-5) ───────► accuracy_markdown ──► experiments/s15_accuracy_table.md ────┐
 compare (section 10) + WHY ──► save_to_library ────► prompts/library.md (v1 kept, v2 added) ┴─► commit, push
                                                     then check_s15_work reads the repo: five PASS lines
```
**Analogy:** a lab notebook. A result that is not written down, dated and signed did not happen. Session 17 will open this library and score your v1 against your v2, so both must be there.

### What the next cell does (find the work repository)
1. **`Path(os.getenv("WORK_REPO") or Path.home() / "Documents" / "genai-course-work")`** builds the path with **pathlib**: the `/` operator joins folder names, and works on Windows, Mac and Linux alike. It is the same folder as Session 14. `os.getenv("WORK_REPO")` reads the `WORK_REPO=` line from your `.env` first; only if it is not set does `or` fall back to `Documents\genai-course-work`. Every session from here on (including Session 17) finds your work repository the same way, so it is set in ONE place.
2. Prints the path and whether it exists. If it does not, it tells you to clone your work repository first.

In [16]:
WORK_REPO = Path(os.getenv("WORK_REPO") or Path.home() / "Documents" / "genai-course-work")   # the same folder as Session 14
print("work repo:", WORK_REPO, "| exists:", WORK_REPO.exists())
if not WORK_REPO.exists():
    print("Clone your work repository first (module02/work_repo_steps.md), then rerun this cell.")

work repo: C:\Users\mynam\Documents\genai-course-work | exists: False
Clone your work repository first (module02/work_repo_steps.md), then rerun this cell.


### What the next cell does (save the table and the two v2 entries)
1. **`SAVE_V2 = False`** is the safety switch. **`WHY`** is a dictionary of two reasons, one per task; fill both with numbers from section 10.
2. **`have_ten = all(... for c in LOAN_CASES for s in ("plain", "cot"))`** is `True` only if all twenty results from section 3 exist.
3. The `if` needs all five conditions: the switch, the folder, the ten cases, the `compare` results from section 10, and both `WHY` lines non-empty (`all(WHY.values())`).
4. For each task, **`compare[task.id][1]["rows"][0]`** takes the v2 result (`[1]` is v2 in the pair), then its first run, and **`save_to_library(...)`** appends a version 2 entry with the scratchpad prompt, its output, its checks and your reason. Older entries are never touched.
5. **`accuracy_markdown(...)`** builds the ten-case table with all four strategy columns (a strategy you did not run for a case shows `-`), and **`save_accuracy_table`** writes it to `experiments/s15_accuracy_table.md`.

In [17]:
SAVE_V2 = False
WHY = {TASK1.id: "",      # YOUR TURN: one line, with numbers from section 10
       TASK2.id: ""}
have_ten = all((c.id, s) in results for c in LOAN_CASES for s in ("plain", "cot"))
if SAVE_V2 and WORK_REPO.exists() and have_ten and compare and all(WHY.values()):
    for task in (TASK1, TASK2):
        run_v2 = compare[task.id][1]["rows"][0]
        save_to_library(WORK_REPO, task.id, 2, with_scratchpad(BASE[task.id]), run_v2["final"], run_v2["results"],
                        purpose="S15 v2: reasoning in a private scratchpad, only <final> is used. " + WHY[task.id])
    table = accuracy_markdown(results, LOAN_CASES, ("plain", "cot", "stepback", "vote x5"), MODEL)
    path = save_accuracy_table(WORK_REPO, table)
    print("saved 2 version-2 entries to", WORK_REPO / "prompts" / "library.md")
    print("saved the accuracy table to", path)
else:
    print("not saved (SAVE_V2 is False, or the work repo, sections 3 and 10 results, or a WHY line is missing)")

not saved (SAVE_V2 is False, or the work repo, sections 3 and 10 results, or a WHY line is missing)


In a terminal (not in this notebook), inside your **work** repository:

```powershell
cd ~\Documents\genai-course-work
git add .
git commit -m "Session 15: accuracy table and prompt library v2"
git push
```
Then run the self-check below. It reads your repository.

### What the next cell does (the self-check)
**`check_s15_work(WORK_REPO)`** reads your work repository (not GitHub) and returns a list of results, each with a name, `passed`, and a `detail` that says how to fix a failure. The loop prints `PASS` or `FAIL` for each, using a *conditional expression* inside `print`, and the fix under each failure.

In [18]:
for r in check_s15_work(WORK_REPO):
    print(("PASS  " if r.passed else "FAIL  ") + r.name)
    if not r.passed and r.detail:
        print("        fix:", r.detail)

FAIL  prompts/library.md exists in your work repo
        fix: run the Session 14 notebook cells first
FAIL  experiments/s15_accuracy_table.md exists
        fix: set SAVE_V2 = True in section 11 and run that cell once


### What you should see
From the first cell: `work repo: C:\Users\you\Documents\genai-course-work | exists: True`. From the save cell, after you switch it on: two `saved ...` lines. Before that: `not saved (...)`, which is correct. From the self-check, when everything is done, five lines:
```
PASS  library still has your version 1 entries
PASS  version 2 entries for at least 2 tasks
PASS  every v2 task still has its v1 entry (history kept)
PASS  experiments/s15_accuracy_table.md exists
PASS  the table has all 10 loan cases
```
If `prompts/library.md` or the table file is missing, you see fewer lines: each missing file gives one `FAIL` with its fix.

| If you see... | It means... | Do this |
|---|---|---|
| `exists: False` | the work repo is not at that path | clone it (`module02/work_repo_steps.md`), or set `WORK_REPO` to where it is |
| `FAIL prompts/library.md exists ... run the Session 14 notebook cells first` | no library from Session 14 | run the Session 14 save cell first, then come back |
| `NameError: name 'compare' is not defined` | section 10 was not run in this kernel | run section 10 (cached answers are free), then this cell |
| `FAIL the table has all 10 loan cases` | section 3 did not finish for every case | re-run section 3, then the save cell (it overwrites the table; Git keeps the history) |

### Check yourself
*Why does the save cell refuse to run while a `WHY` line is empty? What would a reviewer in Session 17 learn from "looks better" that they would not learn from "4.0 vs 3.0 of 4 checks, 2.1x tokens"?*

> **Common mistake:** editing the version 1 entry "to make it pass". The check `every v2 task still has its v1 entry` exists to catch that. History is the point: add v2, keep v1.

## 12. Your decision table

### What and why
Fill this in from **your own numbers**, one word per cell: *use*, *skip*, or *only if* (and say what). It is the other deliverable of the session. This table is what you would bring to a design review: not "CoT is good", but "for this kind of question, on this model, it bought X at Y times the cost".

### The idea: three questions before you pay for thinking
1. **Is there one answer a machine can check?** If yes, you can measure, and you can vote. If not, voting only averages opinions.
2. **Does it chain several steps?** Steps that depend on each other are where working helps. A single lookup gains nothing.
3. **Is the rule already in the prompt?** Thinking cannot recover a rule the model was never told. Teach house rules with context or examples (Session 14).

And if a calculator can do the sum, give the model a calculator (Session 20).

### How to fill it in
Put your section 3 table, your section 4–5 results and your section 7 table next to the grid. Write *use* only where your numbers showed a gain worth its price; *skip* where it bought nothing; *only if* with a condition, for example "only if one wrong answer costs more than Rs 10,000".

| Kind of question | plain | step by step | step back | vote x5 |
|---|---|---|---|---|
| Several rules chained, nothing hidden (L02, L07) | | | | |
| A rule that fails quietly (L05, L10) | | | | |
| Values exactly on a limit (L03, L04, L06) | | | | |
| A formula or a piece of theory to recall (R4) | | | | |
| Classification with your house rules (the complaints) | | | | |
| A calculation a calculator does exactly (S3, the EMI) | | | | |

### Check yourself
*Which row was the hardest to fill from your own numbers, and what one extra experiment would settle it?*

> **Common mistake:** filling the table from the lecture instead of from your screen. If your numbers disagree with the instructor's, yours win, and the disagreement is worth reporting.

## 13. Reflection

### What and why
Writing one or two lines per question turns a run of numbers into something you can explain in an interview. The final sentence is the summary of the whole session in your own words.

Answer questions 1-8 above, one or two lines each. Then one sentence: **when would you pay for thinking, and when would you refuse?**

1.Question 1 — Where did the output tokens go? Who paid for them, and could you see them? What does that mean for the phrase 'think step by step' on a model that already thinks?

Answer:- The output tokens were used for the model’s internal reasoning and final response. The provider charged for those tokens as part of the completion cost. The internal reasoning tokens were not necessarily visible to the user; typically, only the final answer was shown. On a model that already reasons internally, adding “think step by step” may increase token usage, cost, and latency without improving accuracy, so it should be used only when testing shows a real benefit.

Question 2 — Put your predictions next to the table. Did step-by-step change the number of right answers? Did it change the cost? Which of the two moved more, and what does that say about when to pay for it?

Answer:- Step-by-step prompting improved accuracy only if it produced more correct answers in the test table, but it also increased token usage and cost. Compare the change in correct answers with the percentage increase in cost to see which moved more. If cost increased more than accuracy, the extra expense may not be worthwhile. Pay for step-by-step reasoning only when the improvement in accuracy justifies the additional cost and latency.

Question 3 — Read the principles the model wrote. Were they useful, or did they just restate the policy you had already given it? When would you expect step-back to beat step-by-step, and when not?

Answer:- Step-back was useful if the model’s principles helped it choose the correct rule or approach rather than simply repeating the policy already provided. It should outperform step-by-step when a problem requires identifying the right general principle, formula, or strategy before solving it. Step-by-step is more suitable when the rules are already clear but the model needs to apply several steps correctly. If the policy is simple and explicit, step-back may add extra tokens and cost without improving accuracy.

Question 4 — The tables give a cost per 1,000 questions for each strategy. Multiply it for 200,000 loan decisions a month. What would a 5x vote add to the monthly bill, and what size of accuracy gain would justify it? For which kinds of question would you never vote?

Answer:- For 200,000 loan decisions per month, multiply the cost per 1,000 questions by 200 to calculate the monthly bill for each strategy. A 5× vote strategy costs approximately five times as much as a single call, so the additional cost is about 4× the single-call monthly cost. I would justify this expense only if voting produces a meaningful, measurable improvement in accuracy that outweighs the extra cost and latency. I would never use voting for exact calculations that a calculator can handle reliably, or for classifications where the model lacks the required business rules or examples.

Question 5 — If low and high gave the same answer, what did the extra thinking buy you? If they differed, which would you ship, and how would you know from one case that it was not luck?

Answer:- If low and high reasoning produced the same answer, the extra thinking did not change the result for that case and may have increased cost and latency without providing a clear benefit. If they differed, I would ship the answer that matches the correct expected result, not automatically choose the high-reasoning answer. One case is not enough to prove that higher reasoning is better or that the result was not luck. I would test both settings on multiple cases with known correct answers and compare accuracy, token usage, cost, and latency before deciding.

Question 6 — Which change fixed the wrong labels: thinking, or examples? Why can no amount of thinking recover a rule the model was never told? What does that say about choosing between the two?

Answer:- Examples fixed the wrong labels because they taught the model the specific business rules and classification patterns it was missing. Thinking alone cannot recover a rule that was never provided in the prompt or examples. The model may reason carefully but still reach the wrong answer because it lacks the necessary information. Therefore, provide clear rules and representative examples first, then use additional reasoning if accuracy still needs improvement.

Question 7 — Compare the local model's plain and step-by-step rows with the hosted model's on the same cases. Which model gained more from step-by-step, and why? What does that tell you about advice such as 'always add think step by step'?

Answer:- The local model gained more from step-by-step prompting if its accuracy improved more than the hosted model’s on the same test cases. This is because the local model may benefit more from explicit reasoning instructions, while the hosted model may already have strong internal reasoning capabilities. However, the actual results table is needed to confirm which model improved more. This shows that “always add think step by step” is not a universal rule; test each model and use step-by-step prompting only when it measurably improves accuracy enough to justify the extra cost and latency.

Question 8 — Task 2's reply must never reveal the internal score. Did the scratchpad mention it? Who inside the bank may read a scratchpad, where would you store it, and for how long? Would you still keep the curtain if v2 gave no better check results?

Answer:- The scratchpad may mention the internal score, so it must be kept separate from the final reply to prevent confidential information from being exposed. Only authorized bank employees who need it for their work should be allowed to read the scratchpad. It should be stored securely with access controls and retained only for the period required by the bank’s data-retention policy. Yes, I would keep the curtain even if v2 showed no improvement in check results, because separating internal reasoning from the final response protects sensitive information independently of accuracy.

Checkpoint: `uv run pytest tests/test_s15.py` is green, and the self-check above says `PASS` on all five lines. Push, and submit your repository link again.

### Check yourself
*Read your sentence aloud. Does it name a kind of question, a number from your lab, and a cost? If not, rewrite it until it does.*

> **Common mistake:** pushing without running the self-check. Run the last cell of section 11 after `git push`; it must show five `PASS` lines.

---
# Part C · Wrap-up

## C1. Summary
- **Thinking is billed.** Hidden or visible, it is output tokens, seconds and money. On a reasoning model a one-line answer can cost hundreds of tokens (section 2).
- **Measure, do not assume.** The same ten cases, four ways of asking, each read as right or wrong, tokens, seconds and cost per 1,000 (sections 3–5). "Both perfect, step by step dearer" is a finding, not a failure.
- **Match the method to the trap.** Step by step helps when steps chain; step back helps when the trap is choosing or recalling the rule; a vote fixes random slips, never a wrong rule (sections 3–5).
- **The dial is a cost decision.** `reasoning_effort` moves thinking tokens and seconds; whether it moves accuracy is measured per kind of question (section 6).
- **Thinking cannot supply a missing rule** (examples can), and it should not do arithmetic a calculator does exactly (sections 7 and 9).
- **Working stays behind a curtain.** Checks run on `<final>` only; the scratchpad is never shown or stored where a customer can read it (section 10).
- **Evidence is committed.** The ten-case accuracy table and v2 beside v1, with a reason in numbers (section 11).

## C2. The words again

| Term | One-line meaning |
|---|---|
| Reasoning model | Writes hidden working before the visible answer, and bills it as output |
| Reasoning tokens | The output tokens spent on that hidden working |
| Chain of thought | "Think step by step, show each calculation, then FINAL:" |
| FINAL line | The last `FINAL: <answer>` line, the only place the code reads the answer |
| Ground truth | The answer key, worked out independently of the model |
| Step-back prompting | Principles first in a call that may not solve, then solve with them |
| Self-consistency | Several samples above temperature 0, the majority answer wins |
| Temperature and samples | Randomness on purpose, so votes can differ; each draw is a sample |
| Majority vote and agreement | The winning answer and the share of valid votes it got |
| Reasoning effort | The low / medium / high dial on some models: thinking against cost and delay |
| DiskCache | The on-disk memory of answers: re-runs are free and identical |
| Meter and token budget | What you have spent, and what you may spend |
| Rate limit (429) | Per-minute: wait. Daily: stop for today |
| Cost per 1,000 questions | Tokens × price, scaled so finance can compare |
| Scratchpad and `<final>` | Private working your code strips; the only part that ships |
| Prompt version 2 | A new entry beside v1, with a reason in numbers; v1 never edited |

## C3. Interview practice

**1. "What is chain-of-thought prompting, and when does it not help?"**
*Model answer:* Asking the model to write its intermediate steps before the final answer, then reading the answer from a fixed line such as `FINAL:`. It helps when the answer depends on several linked steps that a one-word answer can skip. In my lab, a loan checker on ten cases, I compared it with a plain one-word prompt on the same cases and read accuracy, tokens and seconds side by side. *Honest limit:* it does not help when the rule is missing from the prompt (my complaint classifier needed examples, not thinking), and on a reasoning model that already thinks it can add cost for little or no gain. I would decide per kind of question, from measurements.

**2. "What are reasoning tokens, and how do they change cost and latency?"**
*Model answer:* Tokens a reasoning model writes as hidden working before its visible answer. They are billed as output tokens and take time to generate, even if the user never sees them. In my lab, a one-line `FINAL: 1340` answer reported far more completion tokens than the line itself, and the notebook split them into thinking and visible parts when the provider reported the count. *Honest limit:* not every provider reports the count; when it is estimated from text length or missing, I do not quote it as a measured number.

**3. "Explain self-consistency. When would you use it, and what does it cost?"**
*Model answer:* Ask the same step-by-step question several times at a temperature above zero and take the most common final answer; the share that agrees is a free confidence signal. In my lab I ran five samples on the hardest loan case and compared single-sample accuracy with the majority, and the cost with one call: about five times the tokens and the time. *Honest limit:* it only fixes random slips. If the model applies a wrong rule, all samples agree on the wrong answer. It needs one checkable answer, and it is the wrong tool for arithmetic a calculator does exactly.

**4. "Would you add 'think step by step' to every prompt in production?"**
*Model answer:* No. I would add it where measurements show a gain worth the price, and I would price it at production volume. For example, a cost per 1,000 questions multiplied by 200,000 decisions a month turns a small difference per call into a monthly line item. In my lab I filled a decision table per kind of question: chained rules, boundary values, a missing house rule, an exact calculation. *Honest limit:* ten cases is a small sample, and results change with the model and the day, so I would re-measure on a larger golden set (the Session 17 approach) before a decision.

**5. "How do you stop a model's reasoning from leaking confidential information?"**
*Model answer:* Ask for the working inside `<scratchpad>` tags and the reply inside `<final>` tags; code strips the scratchpad, runs every acceptance check on `<final>` only, and never shows or stores the scratchpad where customers or a wide team can read it. In my lab, the declined-loan SMS passed the "never reveal the score" check on the final part, while the scratchpad itself could mention the score and cutoff. *Honest limit:* tags are an instruction, not a guarantee. The model can still put a secret in `<final>`, so the checks on the final part (and the defences in Session 18) are still needed, and access to stored scratchpads must be controlled and time-limited.

## C4. Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ImportError: ... run 'git pull'` | a course file is missing or old | `git pull`, `uv sync`, **Kernel → Restart** |
| `ready: False` and every model cell says `skipped` | no `.env`, or `BASE_URL` / `API_KEY` / `MODEL` missing | fix `.env` in the course folder (Session 11), restart the kernel |
| `(rate limit: waiting 21s, then trying again)` | the free per-minute limit (about 8,000 tokens a minute) | normal; wait, do not click Stop |
| the cell stops with *"... That is a daily limit, not a per-minute one ..."* | the free daily limit is used up | stop for today; answers already received are in the cache, and re-runs of finished cells are free tomorrow |
| `NameError: name 'trouble' is not defined` (or `'compare'`) | a cell was run before the section it depends on | run the earlier section's cell first (section 4 for `trouble`, section 10 for `compare`); cached answers cost nothing |
| a table shows `none WRONG` or `CUT OFF` | no readable `FINAL:` line, often a reply cut off at the token limit | read the reply text; an unreadable answer counts as wrong; try the case again later or with low effort |
| `effort='low' was not accepted by this model or provider` | the model or provider has no effort dial | a normal finding; move on |
| `Ollama is not reachable: Connection error.` | Ollama is not running | optional; start it (Session 12) or skip section 8 |
| the same numbers every time, even after changing nothing | the DiskCache answered | intended. For a fresh draw, delete `~\.genai_course\s15_cache.json` (this costs tokens) |
| `not saved (SAVE_V2 is False, ...)` | a condition of the save cell is missing | set `SAVE_V2 = True`, fill both `WHY` lines, run sections 3 and 10 first |
| self-check: `FAIL library still has your version 1 entries` | v1 entries are missing or were edited away | restore the Session 14 entries from Git history; never edit v1 to make a check pass |

## C5. What comes next
**Session 16 · Structured prompting:** tags, Jinja templates, a system prompt stored as a file, and a classifier measured on held-out tickets. It imports today's `Lab`, `DiskCache` and `Meter`, so the habits from today (cached re-runs, a meter on every section) continue.

**Keep:**
- your work repository with **both** v1 and v2 entries: Session 17 judges them against each other;
- `experiments/s15_accuracy_table.md` and your decision table: they are your evidence for the interview question "would you add 'think step by step' everywhere?";
- the idea of a checkable answer line (`FINAL:`): golden sets and judges in Session 17 depend on it.